# Kaggle 2×T4：单卡 / DP2 / PP2 / PD 分离 推理架构小规模对比

**按顺序执行，每一步看到 `PASS` 再继续。** 详细说明见 README.md（与本 Notebook 一起交付）。

| 步骤 | 内容 | 预计耗时 | 级别/时间上限 |
|---|---|---|---|
| 0 | 写入项目文件 | <1 min | – |
| 1–2 | 环境检查（双 T4、Internet、/dev/shm）与预算计时 | 1 min | P0 |
| 3 | 安装隔离 venv：vLLM 0.8.5.post1（依赖冻结到 2025-05-20） | 5–10 min | P0 ≤ 60 min |
| 4 | 下载 Qwen2.5-1.5B-Instruct（固定 revision）+ 生成固定 token 负载 | 2 min | P0 |
| 5 | A 单卡 smoke | 3 min | P0 |
| 6 | C PP2 smoke（验证真的是两段流水线） | 4 min | P1 ≤ 30 min |
| 7 | D PD 分离 smoke（验证真实 KV 传输 + 负对照） | 4 min | P2 ≤ 60 min |
| 8 | B DP2 smoke | 4 min | P3 ≤ 30 min |
| 9 | 正式小规模 benchmark（16 个配置） | 30–50 min | 总会话上限 6 h |
| 10 | 报告、图表、ZIP | 1 min | – |
| 11 | **关闭所有进程，然后在 Kaggle 右上角 Stop Session** | – | – |

整套首轮预计 **1–1.5 GPU 小时**。所有结果写入 `/kaggle/working/kaggle-pd-pp-lab/results/`，每个配置完成即落盘，可断点续跑。

In [ ]:
import os, glob, json
LAB = "/kaggle/working/kaggle-pd-pp-lab"
for d in ["scripts/pd_probe", "configs", "results"]:
    os.makedirs(os.path.join(LAB, d), exist_ok=True)
os.chdir(LAB)
VPY = "/tmp/pdlab-venv/bin/python"   # isolated vLLM venv created in step 3
print("working dir:", os.getcwd())

## 0. 写入项目文件（直接运行以下所有单元格，无需修改）
这些单元格把全部脚本写到 `kaggle-pd-pp-lab/` 下。代码与 README 中的项目文件完全一致。

In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/configs/experiments.json
{
  "project": "kaggle-pd-pp-lab",
  "model": {
    "repo_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "fallback_repo_id": "Qwen/Qwen2.5-0.5B-Instruct",
    "served_name": "pdlab",
    "num_layers": 28,
    "num_kv_heads": 2,
    "head_dim": 128,
    "hidden_size": 1536
  },
  "vllm": {
    "version": "0.8.5.post1",
    "exclude_newer": "2025-05-20T00:00:00Z",
    "python": "3.11",
    "engine_env": {"VLLM_USE_V1": "0"},
    "dtype": "float16",
    "max_model_len": 2048,
    "gpu_memory_utilization": 0.8,
    "max_num_seqs": 32,
    "max_num_batched_tokens": 8192,
    "swap_space_gib": 1,
    "enforce_eager": false,
    "seed": 0,
    "extra_args": []
  },
  "kv_transfer": {
    "connector": "PyNcclConnector",
    "kv_buffer_size": 1000000000,
    "kv_port": 14579
  },
  "ports": {"proxy": 8000, "inst0": 8100, "inst1": 8200},
  "workloads": {
    "prefill_heavy": {"input_tokens": 1024, "output_tokens": 32, "num_requests": 40},
    "decode_heavy": {"input_tokens": 128, "output_tokens": 256, "num_requests": 32}
  },
  "smoke": {"prefill_heavy_requests": 2, "decode_heavy_requests": 2, "decode_heavy_output_tokens": 32},
  "concurrency_levels": [2, 8],
  "warmup_requests": 4,
  "max_reps": 3,
  "arch_order": ["A", "C", "D", "B"],
  "timeouts": {"server_start_s": 900, "request_s": 180, "run_s": 900, "stop_s": 90},
  "budget": {
    "total_session_hours_cap": 6.0,
    "level_caps_min": {"P0": 60, "P1": 30, "P2": 60, "P3": 30},
    "note": "Wall-clock from env_check (first cell) approximates GPU-session time, because Kaggle bills the whole session while the accelerator is attached."
  },
  "slo": {
    "rule": "relative_to_A_c2",
    "ttft_factor": 3.0,
    "tpot_factor": 1.25,
    "fallback_abs": {
      "prefill_heavy": {"ttft_ms": 1000, "tpot_ms": 60},
      "decode_heavy": {"ttft_ms": 300, "tpot_ms": 60}
    },
    "note": "Pre-registered before any measurement: per workload, TTFT SLO = 3.0 x A(single GPU) TTFT P50 at concurrency 2 and TPOT SLO = 1.25 x A TPOT P50 at concurrency 2. The absolute fallback is only used when A is missing."
  },
  "gpu_sample_interval_s": 0.5
}


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/common.py
"""Shared helpers: paths, config, atomic JSON, nvidia-smi sampling, budget guard.

Only the standard library is used so that every script can run either with the
Kaggle system Python or with the isolated vLLM virtualenv.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

SCRIPTS_DIR = Path(__file__).resolve().parent
LAB_ROOT = Path(os.environ.get("PDLAB_ROOT", SCRIPTS_DIR.parent)).resolve()
CONFIG_PATH = Path(os.environ.get("PDLAB_CONFIG", LAB_ROOT / "configs" / "experiments.json"))
RESULTS = Path(os.environ.get("PDLAB_RESULTS", LAB_ROOT / "results"))
LOG_DIR = RESULTS / "logs"
STATE_DIR = RESULTS / "state"
ENV_DIR = RESULTS / "env"
SMOKE_DIR = RESULTS / "smoke"
RUNS_DIR = RESULTS / "runs"
WORKLOAD_PATH = RESULTS / "workload" / "workloads.json"
VENV = Path(os.environ.get("PDLAB_VENV", "/tmp/pdlab-venv"))
MODEL_DIR = Path(os.environ.get("PDLAB_MODEL_DIR", "/tmp/pdlab-models"))
HF_HOME = os.environ.get("PDLAB_HF_HOME", "/tmp/pdlab-hf")

ARCH_NAMES = {
    "A": "Single-GPU mixed P+D (GPU0)",
    "B": "DP2: two full replicas + round-robin proxy",
    "C": "PP2: one instance, layers split over GPU0/GPU1",
    "D": "PD disaggregation: prefill GPU0 -> KV (PyNccl) -> decode GPU1",
}
ARCH_LEVEL = {"A": "P0", "C": "P1", "D": "P2", "B": "P3"}


def load_config() -> dict:
    with open(CONFIG_PATH, encoding="utf-8") as f:
        return json.load(f)


def ensure_dirs() -> None:
    for d in (RESULTS, LOG_DIR, STATE_DIR, ENV_DIR, SMOKE_DIR, RUNS_DIR, WORKLOAD_PATH.parent):
        d.mkdir(parents=True, exist_ok=True)


def now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime())


def write_json(path: Path, obj) -> None:
    """Atomic write so a killed session never leaves a half-written result."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def read_json(path: Path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return default


def append_jsonl(path: Path, obj) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    out = []
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        out.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass
    except FileNotFoundError:
        pass
    return out


def run_cmd(cmd: list[str] | str, timeout: float = 60, env: dict | None = None) -> tuple[int, str]:
    try:
        p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
                           timeout=timeout, env=env)
        return p.returncode, (p.stdout or "") + (p.stderr or "")
    except subprocess.TimeoutExpired:
        return -9, f"TIMEOUT after {timeout}s: {cmd}"
    except FileNotFoundError as e:
        return -127, str(e)


def venv_python() -> str:
    p = VENV / "bin" / "python"
    return str(p) if p.exists() else sys.executable


def tail(path: Path, n: int = 40) -> str:
    try:
        with open(path, encoding="utf-8", errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return "<no log>"


# ---------------------------------------------------------------- GPU sampling
def has_nvidia_smi() -> bool:
    return shutil.which("nvidia-smi") is not None


def gpu_query() -> list[dict]:
    if not has_nvidia_smi():
        return []
    rc, out = run_cmd(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                       "--format=csv,noheader,nounits"], timeout=15)
    rows = []
    if rc != 0:
        return rows
    for line in out.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 5:
            continue
        try:
            rows.append({"index": int(parts[0]), "name": parts[1], "mem_used_mib": float(parts[2]),
                         "mem_total_mib": float(parts[3]), "util_pct": float(parts[4])})
        except ValueError:
            continue
    return rows


def gpu_compute_apps() -> list[dict]:
    """Processes holding GPU memory, mapped to GPU index via UUID."""
    if not has_nvidia_smi():
        return []
    rc1, uu = run_cmd(["nvidia-smi", "--query-gpu=index,uuid", "--format=csv,noheader"], timeout=15)
    rc2, apps = run_cmd(["nvidia-smi", "--query-compute-apps=pid,gpu_uuid,used_memory,process_name",
                         "--format=csv,noheader,nounits"], timeout=15)
    if rc1 != 0 or rc2 != 0:
        return []
    idx = {}
    for line in uu.strip().splitlines():
        a = [x.strip() for x in line.split(",")]
        if len(a) == 2:
            idx[a[1]] = int(a[0])
    rows = []
    for line in apps.strip().splitlines():
        a = [x.strip() for x in line.split(",")]
        if len(a) >= 3 and a[0].isdigit():
            rows.append({"pid": int(a[0]), "gpu": idx.get(a[1]), "used_mib": float(a[2]) if a[2].replace('.', '', 1).isdigit() else None,
                         "name": a[3] if len(a) > 3 else ""})
    return rows


class GpuSampler:
    """Background nvidia-smi poller: per-GPU peak memory and mean utilization."""

    def __init__(self, interval_s: float = 0.5):
        self.interval_s = interval_s
        self.samples: list[tuple[float, list[dict]]] = []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)

    def _loop(self):
        while not self._stop.is_set():
            rows = gpu_query()
            if rows:
                self.samples.append((time.time(), rows))
            self._stop.wait(self.interval_s)

    def __enter__(self):
        if has_nvidia_smi():
            self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self._thread.is_alive():
            self._thread.join(timeout=5)

    def summary(self) -> dict:
        per = {}
        for _, rows in self.samples:
            for r in rows:
                d = per.setdefault(str(r["index"]), {"mem_peak_mib": 0.0, "util": []})
                d["mem_peak_mib"] = max(d["mem_peak_mib"], r["mem_used_mib"])
                d["util"].append(r["util_pct"])
        out = {}
        for k, d in per.items():
            u = d["util"]
            out[k] = {"mem_peak_mib": d["mem_peak_mib"],
                      "util_mean_pct": round(sum(u) / len(u), 2) if u else None,
                      "util_max_pct": max(u) if u else None,
                      "busy_frac": round(sum(1 for x in u if x > 0) / len(u), 3) if u else None}
        return {"n_samples": len(self.samples), "per_gpu": out}


# ---------------------------------------------------------------- budget guard
class BudgetExceeded(RuntimeError):
    pass


class Budget:
    """Wall-clock guard. Kaggle bills the whole GPU session, so we measure from the
    first env_check of this session (results/state/budget.json)."""

    def __init__(self, cfg: dict):
        self.cfg = cfg["budget"]
        self.path = STATE_DIR / "budget.json"
        self.state = read_json(self.path, None)
        if self.state is None:
            self.state = {"session_start": time.time(), "session_start_iso": now_iso(), "levels": {}}
            self._save()

    def _save(self):
        write_json(self.path, self.state)

    def reset_session(self):
        self.state["session_start"] = time.time()
        self.state["session_start_iso"] = now_iso()
        self._save()

    def elapsed_h(self) -> float:
        return (time.time() - self.state["session_start"]) / 3600.0

    def check_total(self, force: bool = False):
        cap = self.cfg["total_session_hours_cap"]
        if self.elapsed_h() > cap and not force:
            raise BudgetExceeded(f"Session wall-clock {self.elapsed_h():.2f} h exceeds cap {cap} h; "
                                 f"stop and save results (use --force only if you consciously accept more GPU time).")

    def enter_level(self, level: str, force: bool = False):
        self.check_total(force)
        lv = self.state["levels"].setdefault(level, {"first_start": time.time(), "attempts": 0, "status": "running"})
        lv["attempts"] += 1
        self._save()
        cap_min = self.cfg["level_caps_min"].get(level)
        spent = (time.time() - lv["first_start"]) / 60
        if cap_min is not None and spent > cap_min and lv.get("status") != "pass" and not force:
            raise BudgetExceeded(f"Level {level} debugging time {spent:.1f} min exceeds cap {cap_min} min. "
                                 f"Keep the error evidence in results/ and move on (or --force).")

    def mark_level(self, level: str, status: str, note: str = ""):
        lv = self.state["levels"].setdefault(level, {"first_start": time.time(), "attempts": 0})
        lv["status"] = status
        lv["note"] = note
        lv["updated"] = now_iso()
        lv["spent_min"] = round((time.time() - lv["first_start"]) / 60, 1)
        self._save()

    def report(self) -> str:
        lines = [f"Session started {self.state['session_start_iso']}, elapsed {self.elapsed_h():.2f} h "
                 f"(cap {self.cfg['total_session_hours_cap']} h)"]
        for k, v in sorted(self.state["levels"].items()):
            lines.append(f"  {k}: status={v.get('status')} attempts={v.get('attempts')} "
                         f"spent={(time.time() - v['first_start']) / 60:.1f} min "
                         f"(cap {self.cfg['level_caps_min'].get(k)} min) {v.get('note', '')}")
        return "\n".join(lines)


def percentile(values, q: float):
    """Linear-interpolated percentile (same convention as numpy default); q in [0, 100]."""
    vals = sorted(v for v in values if v is not None)
    if not vals:
        return None
    pos = (len(vals) - 1) * q / 100.0
    lo = int(pos)
    hi = min(lo + 1, len(vals) - 1)
    return vals[lo] + (vals[hi] - vals[lo]) * (pos - lo)


def mean(values):
    vals = [v for v in values if v is not None]
    return sum(vals) / len(vals) if vals else None


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/env_check.py
"""Step 1: environment check (no CUDA context is created; only nvidia-smi is used).

Writes results/env/env_check.json and starts the session budget clock.
Exit code 0 = usable 2xT4 environment, 1 = blocking problem.
"""
from __future__ import annotations

import argparse
import os
import platform
import shutil
import sys
import urllib.request

from common import ENV_DIR, Budget, ensure_dirs, gpu_query, load_config, now_iso, run_cmd, write_json


def disk_free_gb(path: str):
    try:
        return round(shutil.disk_usage(path).free / 1e9, 1)
    except FileNotFoundError:
        return None


def internet_ok(url="https://huggingface.co") -> bool:
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=10)
        return True
    except Exception:
        return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--reset-budget", action="store_true", help="restart the session clock (new Kaggle session)")
    args = ap.parse_args()
    ensure_dirs()
    cfg = load_config()
    budget = Budget(cfg)
    if args.reset_budget:
        budget.reset_session()

    info = {"time": now_iso(), "python": sys.version.split()[0], "platform": platform.platform(),
            "cpu_count": os.cpu_count()}
    gpus = gpu_query()
    info["gpus"] = gpus
    rc, cc = run_cmd(["nvidia-smi", "--query-gpu=index,compute_cap,driver_version,pcie.link.gen.max,pcie.link.width.max",
                      "--format=csv,noheader"], timeout=15)
    info["gpu_details_raw"] = cc.strip()
    rc, smi = run_cmd(["nvidia-smi"], timeout=15)
    info["nvidia_smi_header"] = "\n".join(smi.splitlines()[:4])
    rc, topo = run_cmd(["nvidia-smi", "topo", "-m"], timeout=15)
    info["topology"] = topo.strip()
    try:
        with open("/proc/meminfo") as f:
            info["ram_total_gb"] = round(int(f.readline().split()[1]) / 1e6, 1)
    except Exception:
        info["ram_total_gb"] = None
    info["disk_free_gb"] = {p: disk_free_gb(p) for p in ("/tmp", "/kaggle/working", "/dev/shm")}
    info["internet"] = internet_ok()

    problems, warnings = [], []
    if len(gpus) < 2:
        problems.append(f"Need 2 GPUs, found {len(gpus)}. In Kaggle: Settings -> Accelerator -> 'GPU T4 x2'.")
    elif not all("T4" in g["name"] for g in gpus[:2]):
        warnings.append(f"GPUs are {[g['name'] for g in gpus]}, not T4; results not comparable to the plan.")
    busy = [g for g in gpus if g["mem_used_mib"] > 500]
    if busy:
        warnings.append(f"GPU memory already in use: {[(g['index'], g['mem_used_mib']) for g in busy]} "
                        f"(leftover processes? run scripts/servers.py stop)")
    if not info["internet"]:
        problems.append("No internet. In Kaggle: Settings -> Internet -> On (requires phone-verified account).")
    if (info["disk_free_gb"].get("/tmp") or 0) < 25:
        warnings.append("Less than 25 GB free in /tmp (venv ~8 GB + model ~3.5 GB).")
    shm = info["disk_free_gb"].get("/dev/shm")
    if shm is not None and shm < 1:
        warnings.append(f"/dev/shm only {shm} GB; NCCL SHM transport may fail. "
                        f"The server launcher will set NCCL_SHM_DISABLE=1 for PP/PD in that case.")
    info["problems"], info["warnings"] = problems, warnings
    info["ok"] = not problems
    write_json(ENV_DIR / "env_check.json", info)

    print(info["nvidia_smi_header"])
    for g in gpus:
        print(f"GPU{g['index']}: {g['name']}  mem {g['mem_used_mib']:.0f}/{g['mem_total_mib']:.0f} MiB  util {g['util_pct']}%")
    print("compute_cap / driver / PCIe:", info["gpu_details_raw"].replace("\n", " | "))
    print("topology:\n" + info["topology"])
    print(f"python {info['python']}, cpus {info['cpu_count']}, RAM {info['ram_total_gb']} GB, disk free {info['disk_free_gb']}, internet {info['internet']}")
    for w in warnings:
        print("WARNING:", w)
    for p in problems:
        print("BLOCKING:", p)
    print(budget.report())
    print("ENV_CHECK", "PASS" if info["ok"] else "FAIL")
    sys.exit(0 if info["ok"] else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/install_or_check.py
"""Step 2/3: isolated vLLM virtualenv + pinned model download.

Why an isolated venv: Kaggle's preinstalled torch/transformers move every month;
installing an old vLLM into the system env silently up/downgrades them. `uv` with
`--exclude-newer` resolves every dependency as it existed when vLLM 0.8.5.post1
was released, which keeps the environment reproducible months later.

Usage (system python):
  python3 scripts/install_or_check.py --step env     # venv + vllm (5-10 min, once)
  python3 scripts/install_or_check.py --step model   # model snapshot (~3.1 GB)
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys
import time
from pathlib import Path

from common import ENV_DIR, LOG_DIR, MODEL_DIR, VENV, Budget, BudgetExceeded, ensure_dirs, load_config, read_json, run_cmd, write_json

VERIFY_SNIPPET = r"""
import json, torch
info = {"torch": torch.__version__, "torch_cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(), "device_count": torch.cuda.device_count()}
try:
    info["nccl"] = ".".join(map(str, torch.cuda.nccl.version()))
except Exception as e:
    info["nccl"] = repr(e)
if torch.cuda.is_available():
    info["devices"] = [{"name": torch.cuda.get_device_name(i), "cc": list(torch.cuda.get_device_capability(i))}
                       for i in range(torch.cuda.device_count())]
    try:
        info["p2p_0_1"] = torch.cuda.can_device_access_peer(0, 1) if torch.cuda.device_count() > 1 else None
    except Exception as e:
        info["p2p_0_1"] = repr(e)
import vllm, transformers
info["vllm"] = vllm.__version__
info["transformers"] = transformers.__version__
try:
    import xformers; info["xformers"] = xformers.__version__
except Exception as e:
    info["xformers"] = "missing: " + repr(e)
import sys; info["python"] = sys.version.split()[0]
print("VERIFY_JSON=" + json.dumps(info))
"""


def sh(cmd: list[str], log: Path, timeout: int) -> int:
    print("$", " ".join(cmd), flush=True)
    with open(log, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(cmd)}\n")
        f.flush()
        try:
            p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, timeout=timeout)
            return p.returncode
        except subprocess.TimeoutExpired:
            f.write(f"TIMEOUT after {timeout}s\n")
            return -9


def ensure_uv(log: Path) -> list[str]:
    import shutil
    if shutil.which("uv"):
        return [shutil.which("uv")]
    rc, _ = run_cmd([sys.executable, "-m", "uv", "--version"])
    if rc != 0:
        if sh([sys.executable, "-m", "pip", "install", "-q", "uv"], log, 300) != 0:
            raise RuntimeError("pip install uv failed; see " + str(log))
    return [sys.executable, "-m", "uv"]


def verify(py: str) -> dict:
    rc, out = run_cmd([py, "-c", VERIFY_SNIPPET], timeout=300)
    for line in out.splitlines():
        if line.startswith("VERIFY_JSON="):
            return json.loads(line[len("VERIFY_JSON="):])
    raise RuntimeError("verification failed:\n" + out[-3000:])


def step_env(cfg: dict, args) -> dict:
    log = LOG_DIR / "install.log"
    venv = Path(args.venv)
    py = venv / "bin" / "python"
    version = args.vllm_version or cfg["vllm"]["version"]
    exclude = args.exclude_newer or cfg["vllm"]["exclude_newer"]
    prev = read_json(ENV_DIR / "install.json", {})
    if py.exists() and prev.get("ok") and prev.get("verify", {}).get("vllm") == version and not args.force:
        print(f"venv already OK: vllm {version} at {venv} (skip). Use --force to reinstall.")
        return prev
    t0 = time.time()
    uv = ensure_uv(log)
    if not py.exists():
        if sh(uv + ["venv", "--python", cfg["vllm"]["python"], str(venv)], log, 600) != 0:
            print("uv could not provision Python", cfg["vllm"]["python"], "-> falling back to system python")
            if sh(uv + ["venv", "--python", sys.executable, str(venv)], log, 300) != 0:
                raise RuntimeError("venv creation failed; see " + str(log))
    rc = sh(uv + ["pip", "install", "--python", str(py), f"vllm=={version}", "--exclude-newer", exclude],
            log, args.timeout)
    if rc != 0:
        raise RuntimeError(f"vllm install failed (rc={rc}); see {log}")
    info = verify(str(py))
    rc, freeze = run_cmd(uv + ["pip", "freeze", "--python", str(py)], timeout=120)
    (ENV_DIR / "venv_freeze.txt").write_text(freeze)
    res = {"ok": True, "venv": str(venv), "requested_vllm": version, "exclude_newer": exclude,
           "install_seconds": round(time.time() - t0, 1), "verify": info}
    problems = []
    if not info.get("cuda_available") or info.get("device_count", 0) < 2:
        problems.append("torch in venv does not see 2 CUDA devices")
    if any(d["cc"] != [7, 5] for d in info.get("devices", [])):
        problems.append(f"unexpected compute capability {info.get('devices')}")
    if str(info.get("xformers", "")).startswith("missing"):
        problems.append("xformers missing: V0 on Turing needs the XFormers attention backend")
    res["problems"] = problems
    res["ok"] = info.get("vllm") == version and not any("2 CUDA" in p for p in problems)
    write_json(ENV_DIR / "install.json", res)
    return res


def safetensors_ok(path: Path) -> tuple[bool, str]:
    """Header-consistency check (no torch): 8-byte header length + JSON header whose largest
    data offset must end exactly at the file end. Catches truncated / interrupted downloads."""
    try:
        size = path.stat().st_size
        with open(path, "rb") as f:
            n = int.from_bytes(f.read(8), "little")
            if n <= 0 or n > size:
                return False, f"bad header length {n}"
            header = json.loads(f.read(n))
        end = max(v["data_offsets"][1] for k, v in header.items() if k != "__metadata__")
        ok = 8 + n + end == size
        return ok, f"expected {8 + n + end} bytes, file has {size}"
    except Exception as e:
        return False, repr(e)


def step_model(cfg: dict, args) -> dict:
    py = str(Path(args.venv) / "bin" / "python")
    repo = args.model or cfg["model"]["repo_id"]
    rev = args.revision if args.revision is not None else (cfg["model"]["revision"] if repo == cfg["model"]["repo_id"] else "main")
    target = MODEL_DIR / repo.split("/")[-1]
    code = (
        "from huggingface_hub import snapshot_download\n"
        f"p = snapshot_download({repo!r}, revision={rev!r}, local_dir={str(target)!r}, "
        "allow_patterns=['*.json', '*.safetensors', '*.txt', 'LICENSE'])\n"
        "print('MODEL_PATH=' + p)\n")
    t0 = time.time()
    rc, out = run_cmd([py, "-c", code], timeout=1800)
    if rc != 0:
        raise RuntimeError("model download failed:\n" + out[-3000:])
    cfgj = json.loads((target / "config.json").read_text())
    st = sorted(p.name for p in target.glob("*.safetensors"))
    size_gb = round(sum(p.stat().st_size for p in target.glob("*.safetensors")) / 1e9, 2)
    bad = {}
    for name in st:
        ok, msg = safetensors_ok(target / name)
        if not ok:
            bad[name] = msg
    if bad:
        raise RuntimeError(f"corrupted/truncated weights {bad}; delete {target} and rerun this step")
    res = {"ok": bool(st), "repo_id": repo, "revision": rev, "path": str(target), "safetensors": st,
           "weights_gb": size_gb, "download_seconds": round(time.time() - t0, 1),
           "num_hidden_layers": cfgj.get("num_hidden_layers"), "num_key_value_heads": cfgj.get("num_key_value_heads"),
           "hidden_size": cfgj.get("hidden_size"), "num_attention_heads": cfgj.get("num_attention_heads"),
           "torch_dtype_in_checkpoint": cfgj.get("torch_dtype")}
    write_json(ENV_DIR / "model.json", res)
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--step", choices=["env", "model", "verify"], required=True)
    ap.add_argument("--venv", default=str(VENV))
    ap.add_argument("--vllm-version", default=None, help="override (fallback e.g. 0.9.2 with a different --venv)")
    ap.add_argument("--exclude-newer", default=None)
    ap.add_argument("--model", default=None, help="override repo id (fallback Qwen/Qwen2.5-0.5B-Instruct)")
    ap.add_argument("--revision", default=None)
    ap.add_argument("--timeout", type=int, default=1800)
    ap.add_argument("--force", action="store_true")
    args = ap.parse_args()
    ensure_dirs()
    cfg = load_config()
    budget = Budget(cfg)
    try:
        budget.enter_level("P0", force=args.force)
        if args.step == "env":
            res = step_env(cfg, args)
        elif args.step == "model":
            res = step_model(cfg, args)
        else:
            res = verify(str(Path(args.venv) / "bin" / "python"))
        print(json.dumps(res, indent=2))
        ok = res.get("ok", True)
        print("INSTALL_STEP", args.step, "PASS" if ok else "FAIL")
        sys.exit(0 if ok else 1)
    except BudgetExceeded as e:
        print("BUDGET STOP:", e)
        sys.exit(2)
    except Exception as e:
        budget.mark_level("P0", "error", f"{args.step}: {e}"[:300])
        print("INSTALL_STEP", args.step, "FAIL:", e)
        print(f"Full log: {LOG_DIR / 'install.log'}")
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/prepare_workload.py
"""Build the fixed token-ID workload shared by all four architectures.

Prompts are sent as token-ID lists, so every architecture sees exactly the same
input length (no re-tokenization drift) and the PD lookup buffer can match the
prefill/decode halves of a request by token sequence. Every prompt starts with a
unique request header, so there is no shared prefix between requests.

Run with the venv python (needs transformers):  python scripts/prepare_workload.py
"""
from __future__ import annotations

import argparse
import hashlib
import json
import random

from common import ENV_DIR, WORKLOAD_PATH, ensure_dirs, load_config, now_iso, read_json, write_json

SUBJ = ["The scheduler", "A decode worker", "The KV cache manager", "Each prefill batch", "The router",
        "A pipeline stage", "The block allocator", "The tokenizer service", "An attention kernel",
        "The memory profiler", "A replica", "The load balancer", "The serving engine", "A request queue"]
VERB = ["allocates", "evicts", "transfers", "streams", "batches", "schedules", "copies", "reuses",
        "partitions", "validates", "measures", "throttles", "prefetches", "records"]
OBJ = ["paged blocks", "attention keys", "value tensors", "token budgets", "hidden states", "request metadata",
       "latency samples", "GPU memory", "sequence groups", "the logits buffer", "prompt chunks", "CUDA graphs"]
TAIL = ["before the next iteration", "under heavy load", "when the queue grows", "across two devices",
        "with a strict deadline", "after the warmup phase", "for every new sequence", "at a fixed interval",
        "without extra copies", "while decoding continues", "over the PCIe link", "during peak traffic"]


def make_text(rng: random.Random, n_sentences: int) -> str:
    out = []
    for _ in range(n_sentences):
        out.append(f"{rng.choice(SUBJ)} {rng.choice(VERB)} {rng.randint(2, 512)} {rng.choice(OBJ)} {rng.choice(TAIL)}.")
    return " ".join(out)


class MockTokenizer:
    """Local-test stand-in: one token per whitespace-separated word."""
    name_or_path = "mock-whitespace"

    def encode(self, text, add_special_tokens=False):
        return [abs(hash(w)) % 50000 + 100 for w in text.split()]


def build_prompt(tok, uid: str, n_tokens: int, seed: int) -> list[int]:
    rng = random.Random(seed)
    head = tok.encode(f"Request {uid}. Read the following engineering notes carefully.\n", add_special_tokens=False)
    tail = tok.encode("\nQuestion: Summarize the notes above in plain English.\nAnswer:", add_special_tokens=False)
    body_needed = n_tokens - len(head) - len(tail)
    if body_needed <= 0:
        raise ValueError(f"n_tokens={n_tokens} too small")
    body = []
    n_sent = max(4, body_needed // 12)
    while len(body) < body_needed:
        body = tok.encode(" " + make_text(rng, n_sent), add_special_tokens=False)
        n_sent *= 2
    ids = head + body[:body_needed] + tail
    assert len(ids) == n_tokens
    return ids


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mock-tokenizer", action="store_true")
    ap.add_argument("--force", action="store_true")
    args = ap.parse_args()
    ensure_dirs()
    cfg = load_config()
    if WORKLOAD_PATH.exists() and not args.force:
        print(f"workload exists: {WORKLOAD_PATH} (use --force to rebuild; NOTE rebuilding changes prompts)")
        return
    if args.mock_tokenizer:
        tok = MockTokenizer()
        tok_name = tok.name_or_path
    else:
        from transformers import AutoTokenizer
        model = read_json(ENV_DIR / "model.json")
        if not model:
            raise SystemExit("run install_or_check.py --step model first")
        tok = AutoTokenizer.from_pretrained(model["path"])
        tok_name = model["path"]

    data = {"meta": {"created": now_iso(), "tokenizer": tok_name, "config_workloads": cfg["workloads"],
                     "concurrency_levels": cfg["concurrency_levels"], "max_reps": cfg["max_reps"]},
            "smoke": [], "runs": {}}
    sm = cfg["smoke"]
    seed = 1000
    for i in range(sm["prefill_heavy_requests"]):
        w = cfg["workloads"]["prefill_heavy"]
        data["smoke"].append({"id": f"smoke-ph-{i}", "workload": "prefill_heavy", "max_tokens": w["output_tokens"],
                              "prompt_token_ids": build_prompt(tok, f"smoke-ph-{i}", w["input_tokens"], seed + i)})
    for i in range(sm["decode_heavy_requests"]):
        w = cfg["workloads"]["decode_heavy"]
        data["smoke"].append({"id": f"smoke-dh-{i}", "workload": "decode_heavy", "max_tokens": sm["decode_heavy_output_tokens"],
                              "prompt_token_ids": build_prompt(tok, f"smoke-dh-{i}", w["input_tokens"], seed + 100 + i)})
    seed = 10_000
    for wname, w in cfg["workloads"].items():
        for conc in cfg["concurrency_levels"]:
            for rep in range(1, cfg["max_reps"] + 1):
                key = f"{wname}|c{conc}|r{rep}"
                entry = {"warmup": [], "measure": []}
                # warmup >= concurrency so the first full-size batch (and any first-shape
                # kernel selection / JIT) happens before the timed window
                n_warm = max(cfg["warmup_requests"], conc)
                for phase, n in (("warmup", n_warm), ("measure", w["num_requests"])):
                    for i in range(n):
                        uid = f"{wname[:2]}-c{conc}-r{rep}-{phase[0]}{i}"
                        seed += 1
                        entry[phase].append({"id": uid, "workload": wname, "max_tokens": w["output_tokens"],
                                             "prompt_token_ids": build_prompt(tok, uid, w["input_tokens"], seed)})
                data["runs"][key] = entry
    blob = json.dumps(data["runs"], sort_keys=True).encode()
    data["meta"]["runs_sha256"] = hashlib.sha256(blob).hexdigest()
    lens = {k: sorted({len(x["prompt_token_ids"]) for x in v["measure"]}) for k, v in data["runs"].items()}
    data["meta"]["prompt_lengths"] = lens
    write_json(WORKLOAD_PATH, data)
    n = sum(len(v["warmup"]) + len(v["measure"]) for v in data["runs"].values())
    print(f"wrote {WORKLOAD_PATH}: {n} run prompts + {len(data['smoke'])} smoke prompts, sha256={data['meta']['runs_sha256'][:16]}")
    for k, v in lens.items():
        print(f"  {k}: input lengths {v}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/client.py
"""Streaming /v1/completions client, closed-loop load generator and per-run metrics.

Definitions (client side, wall clock = time.perf_counter):
  TTFT  : request sent -> first SSE event carrying a generated token (choices non-empty).
  ITL   : gap between consecutive token events (pooled over all requests).
          Exact per-token only when events == output tokens; that fraction is reported.
  TPOT  : per request (E2E - TTFT) / (output_tokens - 1).
  E2E   : request sent -> stream finished.
A request is successful only if HTTP 200, [DONE] seen, usage present, prompt_tokens and
completion_tokens equal the requested values, and the generated text is non-blank.
"""
from __future__ import annotations

import asyncio
import json
import time

import aiohttp

from common import mean, percentile


async def one_request(session: aiohttp.ClientSession, url: str, model: str, item: dict, timeout_s: float) -> dict:
    payload = {"model": model, "prompt": item["prompt_token_ids"], "max_tokens": item["max_tokens"],
               "temperature": 0.0, "ignore_eos": True, "stream": True,
               "stream_options": {"include_usage": True}}
    rec = {"id": item["id"], "ok": False, "expected_prompt_tokens": len(item["prompt_token_ids"]),
           "expected_output_tokens": item["max_tokens"], "t_send_epoch": time.time()}
    t0 = time.perf_counter()
    chunk_times, texts = [], []
    usage, saw_done, status = None, False, None
    try:
        async with session.post(url + "/v1/completions", json=payload, headers={"X-Request-Id": item["id"]},
                                timeout=aiohttp.ClientTimeout(total=timeout_s)) as r:
            status = r.status
            if r.status != 200:
                raise RuntimeError(f"HTTP {r.status}: {(await r.text())[:300]}")
            buf = b""
            async for chunk in r.content.iter_any():
                buf += chunk
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    line = line.strip()
                    if not line.startswith(b"data:"):
                        continue
                    data = line[5:].strip()
                    if data == b"[DONE]":
                        saw_done = True
                        continue
                    ev = json.loads(data)
                    if ev.get("usage"):
                        usage = ev["usage"]
                    # vLLM streams one event per generated token, even when the token's text
                    # delta is empty (e.g. an incomplete UTF-8 byte); every choice event counts
                    for ch in ev.get("choices") or []:
                        chunk_times.append(time.perf_counter())
                        texts.append(ch.get("text") or "")
        t_end = time.perf_counter()
        text = "".join(texts)
        rec.update({"http_status": status, "n_chunks": len(chunk_times), "text_head": text[:120],
                    "prompt_tokens": (usage or {}).get("prompt_tokens"),
                    "output_tokens": (usage or {}).get("completion_tokens")})
        problems = []
        if not saw_done:
            problems.append("no [DONE]")
        if usage is None:
            problems.append("no usage")
        if rec["prompt_tokens"] != rec["expected_prompt_tokens"]:
            problems.append(f"prompt_tokens {rec['prompt_tokens']} != {rec['expected_prompt_tokens']}")
        if rec["output_tokens"] != rec["expected_output_tokens"]:
            problems.append(f"output_tokens {rec['output_tokens']} != {rec['expected_output_tokens']}")
        if not chunk_times or not text.strip():
            problems.append("empty output")
        if problems:
            raise RuntimeError("; ".join(problems))
        ttft = (chunk_times[0] - t0) * 1000
        e2e = (t_end - t0) * 1000
        n_out = rec["output_tokens"]
        rec.update({"ok": True, "ttft_ms": round(ttft, 3), "e2e_ms": round(e2e, 3),
                    "tpot_ms": round((e2e - ttft) / (n_out - 1), 4) if n_out > 1 else None,
                    "itl_ms": [round((b - a) * 1000, 3) for a, b in zip(chunk_times, chunk_times[1:])],
                    "chunks_eq_tokens": len(chunk_times) == n_out, "text": text})
    except Exception as e:
        rec.update({"ok": False, "http_status": status, "error": f"{type(e).__name__}: {e}"[:500],
                    "e2e_ms": round((time.perf_counter() - t0) * 1000, 3)})
    rec["t_end_epoch"] = time.time()
    return rec


async def run_closed_loop(url: str, model: str, items: list[dict], concurrency: int, timeout_s: float) -> tuple[list[dict], float, float, float]:
    """Keep exactly `concurrency` requests in flight until all items are done."""
    queue = list(items)
    results: list[dict] = []
    conn = aiohttp.TCPConnector(limit=0, force_close=False)
    async with aiohttp.ClientSession(connector=conn) as session:
        async def worker():
            while queue:
                it = queue.pop(0)
                results.append(await one_request(session, url, model, it, timeout_s))
        t_start_epoch = time.time()
        t0 = time.perf_counter()
        await asyncio.gather(*[worker() for _ in range(concurrency)])
        wall = time.perf_counter() - t0
    order = {it["id"]: i for i, it in enumerate(items)}
    results.sort(key=lambda r: order[r["id"]])
    return results, wall, t_start_epoch, time.time()


def summarize(results: list[dict], wall_s: float) -> dict:
    ok = [r for r in results if r["ok"]]
    itl = [x for r in ok for x in r["itl_ms"]]
    out_tok = sum(r["output_tokens"] for r in ok)
    in_tok = sum(r["prompt_tokens"] for r in ok)

    def pct(vals, q):
        v = percentile(vals, q)
        return round(v, 3) if v is not None else None

    ttft = [r["ttft_ms"] for r in ok]
    tpot = [r["tpot_ms"] for r in ok if r["tpot_ms"] is not None]
    e2e = [r["e2e_ms"] for r in ok]
    m = mean(tpot)
    return {
        "n_requests": len(results), "n_ok": len(ok), "success_rate": round(len(ok) / len(results), 4) if results else 0,
        "wall_s": round(wall_s, 3),
        "req_per_s": round(len(ok) / wall_s, 4), "output_tok_per_s": round(out_tok / wall_s, 2),
        "input_tok_per_s": round(in_tok / wall_s, 2),
        "ttft_p50_ms": pct(ttft, 50), "ttft_p95_ms": pct(ttft, 95), "ttft_mean_ms": round(mean(ttft), 3) if ttft else None,
        "tpot_mean_ms": round(m, 3) if m is not None else None, "tpot_p50_ms": pct(tpot, 50), "tpot_p95_ms": pct(tpot, 95),
        "itl_p50_ms": pct(itl, 50), "itl_p95_ms": pct(itl, 95), "itl_p99_ms": pct(itl, 99), "itl_max_ms": round(max(itl), 3) if itl else None,
        "n_itl_samples": len(itl),
        "e2e_p50_ms": pct(e2e, 50), "e2e_p95_ms": pct(e2e, 95),
        "chunks_eq_tokens_frac": round(sum(r["chunks_eq_tokens"] for r in ok) / len(ok), 4) if ok else None,
        "errors": sorted({r.get("error", "")[:160] for r in results if not r["ok"]})[:5],
    }


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/proxy.py
"""Minimal aiohttp proxy used by B (DP2) and D (PD disaggregation).

rr mode : round-robin each /v1/completions request to one of N full replicas.
pd mode : same flow as vLLM's official benchmarks/disagg_benchmarks/disagg_prefill_proxy_server.py
          (v0.8.5): send the request to the prefill instance with max_tokens=1 and wait for it
          to finish (the producer's KV is now in the PyNccl lookup buffer), then send the
          ORIGINAL request to the decode instance and stream its SSE output back unchanged.
          Unlike the official proxy, a failed prefill returns HTTP 502 instead of silently
          continuing (which would make the decode instance recompute the prompt).

Per-request timings are appended to --log (JSONL) so TTFT can be decomposed.
"""
from __future__ import annotations

import argparse
import asyncio
import itertools
import json
import time

from aiohttp import ClientSession, ClientTimeout, TCPConnector, web


def build_app(mode: str, backends: list[str], prefill: str | None, decode: str | None, log_path: str | None):
    rr = itertools.cycle(range(len(backends))) if backends else None
    state: dict = {}
    log_f = open(log_path, "a", buffering=1, encoding="utf-8") if log_path else None

    def log(rec):
        if log_f:
            log_f.write(json.dumps(rec) + "\n")

    async def on_startup(app):
        state["session"] = ClientSession(connector=TCPConnector(limit=0), timeout=ClientTimeout(total=None))

    async def on_cleanup(app):
        await state["session"].close()
        if log_f:
            log_f.close()

    async def stream_back(request, url, body, headers, rec, t_ref):
        sess: ClientSession = state["session"]
        async with sess.post(url, json=body, headers=headers) as r:
            resp = web.StreamResponse(status=r.status)
            resp.content_type = r.headers.get("Content-Type", "text/event-stream").split(";")[0]
            resp.headers["X-PDLab-Upstream"] = url
            await resp.prepare(request)
            first = True
            async for chunk in r.content.iter_any():
                if first:
                    rec["upstream_first_byte_ms"] = round((time.perf_counter() - t_ref) * 1000, 3)
                    first = False
                await resp.write(chunk)
            await resp.write_eof()
            rec["upstream_status"] = r.status
            return resp

    def fwd_headers(request):
        h = {}
        for k in ("X-Request-Id", "Authorization"):
            if k in request.headers:
                h[k] = request.headers[k]
        return h

    async def completions(request: web.Request):
        t0 = time.perf_counter()
        body = await request.json()
        rid = request.headers.get("X-Request-Id", "")
        rec = {"ts": time.time(), "rid": rid, "mode": mode}
        try:
            if mode == "rr":
                i = next(rr)
                rec["backend"] = i
                resp = await stream_back(request, backends[i] + "/v1/completions", body, fwd_headers(request), rec, t0)
            else:
                pbody = dict(body)
                pbody["max_tokens"] = 1
                pbody["stream"] = False
                pbody.pop("stream_options", None)
                async with state["session"].post(prefill + "/v1/completions", json=pbody,
                                                 headers=fwd_headers(request)) as pr:
                    ptext = await pr.text()
                    rec["prefill_status"] = pr.status
                t1 = time.perf_counter()
                rec["prefill_ms"] = round((t1 - t0) * 1000, 3)
                if pr.status != 200:
                    rec["error"] = "prefill failed: " + ptext[:300]
                    log(rec)
                    return web.json_response({"error": rec["error"]}, status=502)
                try:
                    rec["prefill_prompt_tokens"] = json.loads(ptext).get("usage", {}).get("prompt_tokens")
                except Exception:
                    pass
                resp = await stream_back(request, decode + "/v1/completions", body, fwd_headers(request), rec, t1)
                if "upstream_first_byte_ms" in rec:
                    rec["decode_first_byte_ms"] = rec.pop("upstream_first_byte_ms")
            rec["total_ms"] = round((time.perf_counter() - t0) * 1000, 3)
            log(rec)
            return resp
        except (asyncio.CancelledError, ConnectionResetError) as e:
            rec["error"] = f"client disconnected: {type(e).__name__}"
            log(rec)
            raise
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
            log(rec)
            return web.json_response({"error": rec["error"]}, status=502)

    async def health(request):
        sess: ClientSession = state["session"]
        targets = backends if mode == "rr" else [prefill, decode]
        for t in targets:
            try:
                async with sess.get(t + "/health", timeout=ClientTimeout(total=5)) as r:
                    if r.status != 200:
                        return web.Response(status=503, text=f"{t} -> {r.status}")
            except Exception as e:
                return web.Response(status=503, text=f"{t} -> {e}")
        return web.Response(text="ok")

    async def models(request):
        t = backends[0] if mode == "rr" else decode
        async with state["session"].get(t + "/v1/models") as r:
            return web.Response(status=r.status, text=await r.text(), content_type="application/json")

    app = web.Application(client_max_size=64 * 1024 * 1024)
    app.on_startup.append(on_startup)
    app.on_cleanup.append(on_cleanup)
    app.router.add_post("/v1/completions", completions)
    app.router.add_get("/health", health)
    app.router.add_get("/v1/models", models)
    return app


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["rr", "pd"], required=True)
    ap.add_argument("--port", type=int, default=8000)
    ap.add_argument("--backends", default="", help="comma-separated base URLs (rr)")
    ap.add_argument("--prefill", default=None)
    ap.add_argument("--decode", default=None)
    ap.add_argument("--log", default=None)
    a = ap.parse_args()
    backends = [b.rstrip("/") for b in a.backends.split(",") if b]
    if a.mode == "rr" and not backends:
        ap.error("--backends required for rr")
    if a.mode == "pd" and not (a.prefill and a.decode):
        ap.error("--prefill and --decode required for pd")
    app = build_app(a.mode, backends, a.prefill and a.prefill.rstrip("/"), a.decode and a.decode.rstrip("/"), a.log)
    web.run_app(app, host="127.0.0.1", port=a.port, access_log=None, print=lambda *x: print(*x, flush=True))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/evidence.py
"""Evidence extraction from server logs, PD probe logs and proxy logs.

Log strings are those of vLLM 0.8.5 V0:
  "Initializing a V0 LLM engine (v0.8.5.post1) with config: ... tensor_parallel_size=1, pipeline_parallel_size=2, ..."
  "Model loading took %.4f GiB and %.6f seconds"   (once per worker / pipeline stage)
  "Using XFormers backend."
  "Initializing PyNcclConfig under kv_transfer_config ..."
  "[rankN]: Failed to receive all KVs and hidden states, redo model forwarding."   (WARNING)
  "[rankN]: Successfully received all KVs and hidden states, skip model forwarding." (DEBUG)
"""
from __future__ import annotations

import re
from pathlib import Path

from common import mean, percentile, read_jsonl

PATTERNS = {
    "engine_v0": re.compile(r"Initializing a V0 LLM engine \(v([^)]+)\)"),
    "engine_v1": re.compile(r"Initializing a V1 LLM engine"),
    "tp": re.compile(r"tensor_parallel_size=(\d+)"),
    "pp": re.compile(r"pipeline_parallel_size=(\d+)"),
    "model_loading_gib": re.compile(r"Model loading took ([\d.]+) ?GiB"),
    "xformers": re.compile(r"Using XFormers backend"),
    "attn_backend": re.compile(r"Using (\w[\w ]*?) backend\."),
    "pynccl_kv": re.compile(r"Initializing PyNcclConfig under kv_transfer_config"),
    "kv_fail": re.compile(r"Failed to receive all KVs and hidden states"),
    "kv_success": re.compile(r"Successfully received all KVs and hidden states"),
    "cuda_graph": re.compile(r"Graph capturing finished in ([\d.]+) sec"),
    "kv_cache_blocks": re.compile(r"# (?:cuda|CUDA|GPU) blocks: (\d+)"),
    "error": re.compile(r"(Traceback \(most recent call last\)|\bERROR\b|CUDA out of memory)"),
}


def read_from(path: str | Path, offset: int = 0) -> str:
    try:
        with open(path, encoding="utf-8", errors="replace") as f:
            f.seek(offset)
            return f.read()
    except FileNotFoundError:
        return ""


def log_offsets(state: dict) -> dict:
    out = {}
    for p in state.get("procs", []):
        try:
            out[p["name"]] = Path(p["log"]).stat().st_size
        except FileNotFoundError:
            out[p["name"]] = 0
    return out


def scan_text(text: str) -> dict:
    r = {}
    m = PATTERNS["engine_v0"].search(text)
    r["engine"] = f"V0 {m.group(1)}" if m else ("V1" if PATTERNS["engine_v1"].search(text) else None)
    for k in ("tp", "pp"):
        m = PATTERNS[k].search(text)
        r[k] = int(m.group(1)) if m else None
    r["model_loading_gib"] = [float(x) for x in PATTERNS["model_loading_gib"].findall(text)]
    r["attention_backends"] = sorted(set(PATTERNS["attn_backend"].findall(text)))
    r["pynccl_kv_init"] = bool(PATTERNS["pynccl_kv"].search(text))
    r["kv_fail_count"] = len(PATTERNS["kv_fail"].findall(text))
    r["kv_success_count"] = len(PATTERNS["kv_success"].findall(text))
    r["cuda_graph_s"] = [float(x) for x in PATTERNS["cuda_graph"].findall(text)]
    r["kv_cache_blocks"] = [int(x) for x in PATTERNS["kv_cache_blocks"].findall(text)]
    r["error_lines"] = [ln[:300] for ln in text.splitlines() if PATTERNS["error"].search(ln)][:10]
    return r


def scan_server_logs(state: dict, offsets: dict | None = None) -> dict:
    """Per process: scan its log from `offsets[name]` (or from this launch)."""
    out = {}
    for p in state.get("procs", []):
        off = (offsets or {}).get(p["name"], p.get("log_offset", 0))
        out[p["name"]] = scan_text(read_from(p["log"], off))
    return out


def _window(recs, t0, t1):
    return [r for r in recs if t0 <= r.get("ts", 0) <= t1]


def pd_probe_stats(producer_log: str | None, consumer_log: str | None, t0: float, t1: float) -> dict:
    cons = _window(read_jsonl(consumer_log), t0, t1) if consumer_log else []
    prod = _window(read_jsonl(producer_log), t0, t1) if producer_log else []
    sel = [r for r in cons if r.get("ev") == "select"]
    hits = [r for r in sel if r.get("hit")]
    rb = [r for r in cons if r.get("ev") == "recv_batch"]
    ins = [r for r in prod if r.get("ev") == "insert"]
    sb = [r for r in prod if r.get("ev") == "send_batch"]
    kv_bytes = [r.get("bytes_k", 0) + r.get("bytes_v", 0) for r in hits]
    hid_bytes = [r.get("bytes_hidden", 0) for r in hits]
    tokens = [r.get("matched_tokens", 0) for r in hits]
    sel_ms = [r["ms"] for r in hits]

    def r3(x):
        return round(x, 3) if x is not None else None
    total_bytes = sum(kv_bytes) + sum(hid_bytes)
    total_ms = sum(sel_ms)
    return {
        "n_select": len(sel), "n_select_hit": len(hits), "n_select_miss": len(sel) - len(hits),
        "n_recv_batches": len(rb), "n_recv_bypass_true": sum(1 for r in rb if r.get("bypass_model_exec")),
        "n_recv_bypass_false": sum(1 for r in rb if not r.get("bypass_model_exec")),
        "n_producer_inserts": len(ins), "producer_send_batch_ms_mean": r3(mean([r["ms"] for r in sb])),
        "kv_bytes_total": sum(kv_bytes), "hidden_bytes_total": sum(hid_bytes),
        "kv_bytes_per_req_mean": r3(mean(kv_bytes)), "kv_bytes_per_token": r3(sum(kv_bytes) / sum(tokens)) if sum(tokens) else None,
        "transfer_ms_p50": r3(percentile(sel_ms, 50)), "transfer_ms_p95": r3(percentile(sel_ms, 95)),
        "transfer_ms_mean": r3(mean(sel_ms)),
        "recv_batch_ms_p50": r3(percentile([r["ms"] for r in rb], 50)),
        "effective_GBps": r3(total_bytes / (total_ms / 1000) / 1e9) if total_ms > 0 else None,
        "k_shape_example": hits[0].get("k_shape") if hits else None,
        "mock": any(r.get("mock") for r in cons),
    }


def proxy_stats(proxy_log: str | None, t0: float, t1: float) -> dict:
    recs = _window(read_jsonl(proxy_log), t0, t1) if proxy_log else []
    if not recs:
        return {"n": 0}

    def pc(key, q):
        v = percentile([r[key] for r in recs if r.get(key) is not None], q)
        return round(v, 3) if v is not None else None
    out = {"n": len(recs), "n_error": sum(1 for r in recs if r.get("error"))}
    if recs[0].get("mode") == "pd":
        for k in ("prefill_ms", "decode_first_byte_ms", "total_ms"):
            out[f"{k}_p50"], out[f"{k}_p95"] = pc(k, 50), pc(k, 95)
    else:
        b = {}
        for r in recs:
            b[str(r.get("backend"))] = b.get(str(r.get("backend")), 0) + 1
        out["per_backend"] = b
        out["upstream_first_byte_ms_p50"] = pc("upstream_first_byte_ms", 50)
    return out


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/servers.py
"""Start / stop / inspect the serving processes for one architecture.

  A  single instance, CUDA_VISIBLE_DEVICES=0                      client -> :8100
  B  two full instances (GPU0 :8100, GPU1 :8200) + rr proxy       client -> :8000
  C  one instance, CUDA_VISIBLE_DEVICES=0,1, --pipeline-parallel-size 2 (mp backend)
                                                                   client -> :8100
  D  prefill/kv_producer GPU0 :8100 + decode/kv_consumer GPU1 :8200 (PyNcclConnector)
     + pd proxy                                                    client -> :8000

All vLLM instances share the same flags (dtype, max_model_len, memory fraction,
max_num_seqs, max_num_batched_tokens, chunked prefill OFF, prefix caching OFF, seed).

CLI:  python scripts/servers.py start --arch D [--mock]
      python scripts/servers.py stop
      python scripts/servers.py status
"""
from __future__ import annotations

import argparse
import json
import os
import signal
import subprocess
import sys
import time
import urllib.request
from dataclasses import asdict, dataclass, field
from pathlib import Path

from common import (ENV_DIR, LOG_DIR, SCRIPTS_DIR, STATE_DIR, HF_HOME, VENV, ensure_dirs, gpu_compute_apps, gpu_query,
                    load_config, now_iso, read_json, run_cmd, tail, venv_python, write_json)

STATE_FILE = STATE_DIR / "servers.json"
IS_WIN = os.name == "nt"


@dataclass
class ProcSpec:
    name: str
    cmd: list[str]
    env: dict
    log: str
    health: str
    gpus: str = ""
    role: str = ""
    extra: dict = field(default_factory=dict)


def model_path(cfg) -> str:
    m = read_json(ENV_DIR / "model.json")
    return m["path"] if m else cfg["model"]["repo_id"]


def vllm_cmd(cfg, port: int, extra: list[str]) -> list[str]:
    v = cfg["vllm"]
    cmd = [venv_python(), "-m", "vllm.entrypoints.openai.api_server",
           "--model", model_path(cfg), "--served-model-name", cfg["model"]["served_name"],
           "--host", "127.0.0.1", "--port", str(port),
           "--dtype", v["dtype"], "--max-model-len", str(v["max_model_len"]),
           "--gpu-memory-utilization", str(v["gpu_memory_utilization"]),
           "--max-num-seqs", str(v["max_num_seqs"]),
           "--max-num-batched-tokens", str(v["max_num_batched_tokens"]),
           "--no-enable-chunked-prefill", "--no-enable-prefix-caching",
           "--swap-space", str(v.get("swap_space_gib", 4)),
           "--seed", str(v["seed"]), "--disable-log-requests"]
    if v.get("enforce_eager"):
        cmd.append("--enforce-eager")
    return cmd + list(v.get("extra_args", [])) + extra


def base_env(cfg, gpus: str, run_tag: str) -> dict:
    env = dict(os.environ)
    env.update(cfg["vllm"].get("engine_env", {}))
    env["CUDA_VISIBLE_DEVICES"] = gpus
    env["HF_HOME"] = HF_HOME
    env["HF_HUB_OFFLINE"] = "1" if read_json(ENV_DIR / "model.json") else "0"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["VLLM_LOGGING_LEVEL"] = env.get("PDLAB_VLLM_LOG_LEVEL", "INFO")
    env["PYTHONUNBUFFERED"] = "1"
    env["PDLAB_RUN_TAG"] = run_tag
    env.pop("PYTHONPATH", None)
    shm = read_json(ENV_DIR / "env_check.json", {}).get("disk_free_gb", {}).get("/dev/shm")
    if shm is not None and shm < 1:
        env["NCCL_SHM_DISABLE"] = "1"
    if os.environ.get("PDLAB_NCCL_SAFE") == "1":  # documented fallback for NCCL transport problems
        env["NCCL_P2P_DISABLE"] = "1"
        env["NCCL_SHM_DISABLE"] = "1"
    return env


def build_specs(arch: str, cfg: dict, mock: bool = False, consumer_debug: bool = False) -> tuple[list[ProcSpec], str]:
    p = cfg["ports"]
    tag = f"{arch}-{int(time.time())}"
    u0, u1, up = f"http://127.0.0.1:{p['inst0']}", f"http://127.0.0.1:{p['inst1']}", f"http://127.0.0.1:{p['proxy']}"
    probe_dir = LOG_DIR / "pd_probe"

    def inst(name, port, gpus, extra, role="", probe=None, log_level=None):
        if mock:
            cmd = [sys.executable, str(SCRIPTS_DIR / "mock_vllm_server.py"), "--port", str(port), "--role", role or "mixed"]
            env = dict(os.environ)
            env["PYTHONPATH"] = str(SCRIPTS_DIR)
        else:
            cmd = vllm_cmd(cfg, port, extra)
            env = base_env(cfg, gpus, tag)
        if log_level:
            env["VLLM_LOGGING_LEVEL"] = log_level
        if probe:
            probe_dir.mkdir(parents=True, exist_ok=True)
            env["PDLAB_PROBE_LOG"] = str(probe_dir / f"{probe}.jsonl")
            if not mock:
                env["PYTHONPATH"] = str(SCRIPTS_DIR / "pd_probe")
        return ProcSpec(name, cmd, env, str(LOG_DIR / f"server_{arch}_{name}.log"), f"http://127.0.0.1:{port}/health",
                        gpus=gpus, role=role, extra={"port": port, "probe_log": env.get("PDLAB_PROBE_LOG")})

    def proxy(args):
        env = dict(os.environ)
        env.pop("CUDA_VISIBLE_DEVICES", None)
        return ProcSpec("proxy", [venv_python() if not mock else sys.executable, str(SCRIPTS_DIR / "proxy.py"),
                                  "--port", str(p["proxy"])] + args + ["--log", str(LOG_DIR / f"proxy_{arch}.jsonl")],
                        env, str(LOG_DIR / f"server_{arch}_proxy.log"), f"{up}/health", role="proxy",
                        extra={"port": p["proxy"], "proxy_log": str(LOG_DIR / f"proxy_{arch}.jsonl")})

    if arch == "A":
        return [inst("gpu0", p["inst0"], "0", [])], u0
    if arch == "B":
        return [inst("gpu0", p["inst0"], "0", []), inst("gpu1", p["inst1"], "1", []),
                proxy(["--mode", "rr", "--backends", f"{u0},{u1}"])], up
    if arch == "C":
        return [inst("pp2", p["inst0"], "0,1", ["--pipeline-parallel-size", "2", "--tensor-parallel-size", "1",
                                                 "--distributed-executor-backend", "mp"])], u0
    if arch == "D":
        kv = cfg["kv_transfer"]

        def ktc(role, rank):
            return json.dumps({"kv_connector": kv["connector"], "kv_role": role, "kv_rank": rank, "kv_parallel_size": 2,
                               "kv_buffer_size": kv["kv_buffer_size"], "kv_ip": "127.0.0.1", "kv_port": kv["kv_port"]})
        return [inst("prefill", p["inst0"], "0", ["--kv-transfer-config", ktc("kv_producer", 0)], role="producer",
                     probe="producer"),
                inst("decode", p["inst1"], "1", ["--kv-transfer-config", ktc("kv_consumer", 1)], role="consumer",
                     probe="consumer", log_level="DEBUG" if consumer_debug else None),
                proxy(["--mode", "pd", "--prefill", u0, "--decode", u1])], up
    raise ValueError(f"unknown arch {arch}")


def _http_ok(url: str, timeout=3) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


def start(arch: str, cfg: dict, mock: bool = False, consumer_debug: bool = False, timeout_s: float | None = None) -> dict:
    ensure_dirs()
    stop(cfg, quiet=True, wait_gpu=not mock)
    specs, endpoint = build_specs(arch, cfg, mock, consumer_debug)
    timeout_s = timeout_s or cfg["timeouts"]["server_start_s"]
    procs, state = [], {"arch": arch, "mock": mock, "endpoint": endpoint, "started": now_iso(), "t_launch": time.time(), "procs": []}
    for s in specs:
        Path(s.log).parent.mkdir(parents=True, exist_ok=True)
        logf = open(s.log, "a", encoding="utf-8")
        logf.write(f"\n===== {now_iso()} START {arch}/{s.name}: {' '.join(s.cmd)}\n")
        logf.flush()
        kw = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if IS_WIN else {"start_new_session": True}
        pr = subprocess.Popen(s.cmd, env=s.env, stdout=logf, stderr=subprocess.STDOUT, **kw)
        procs.append((s, pr))
        d = asdict(s)
        d.pop("env")
        d["pid"] = pr.pid
        d["log_offset"] = os.path.getsize(s.log)
        state["procs"].append(d)
        write_json(STATE_FILE, state)
        print(f"launched {s.name:8s} pid={pr.pid} gpus={s.gpus or '-'} log={s.log}", flush=True)
    # vLLM instances first, proxy last
    deadline = time.time() + timeout_s
    pending = list(procs)
    last_print = 0
    while pending:
        for s, pr in list(pending):
            if pr.poll() is not None:
                state["status"] = "failed"
                state["error"] = f"{s.name} exited with code {pr.returncode}"
                write_json(STATE_FILE, state)
                msg = f"{s.name} exited early (code {pr.returncode}). Last log lines:\n{tail(Path(s.log), 60)}"
                stop(cfg, quiet=True)
                raise RuntimeError(msg)
            if _http_ok(s.health):
                pending.remove((s, pr))
                print(f"ready   {s.name:8s} after {time.time() - state['t_launch']:.0f}s", flush=True)
        if time.time() > deadline:
            names = [s.name for s, _ in pending]
            logs = "\n".join(f"--- {s.name}\n{tail(Path(s.log), 30)}" for s, _ in pending)
            stop(cfg, quiet=True)
            raise TimeoutError(f"servers {names} not healthy after {timeout_s}s\n{logs}")
        if time.time() - last_print > 30 and pending:
            last_print = time.time()
            print(f"  waiting for {[s.name for s, _ in pending]} ({time.time() - state['t_launch']:.0f}s)", flush=True)
        time.sleep(0.5 if mock else 3)
    state["status"] = "ready"
    state["startup_s"] = round(time.time() - state["t_launch"], 1)
    state["gpu_after_start"] = gpu_query()
    state["gpu_apps_after_start"] = gpu_compute_apps()
    write_json(STATE_FILE, state)
    return state


def alive(state: dict | None = None) -> bool:
    state = state or read_json(STATE_FILE)
    if not state:
        return False
    for p in state.get("procs", []):
        if not _pid_alive(p["pid"]):
            return False
    return True


def _pid_alive(pid: int) -> bool:
    if IS_WIN:
        rc, out = run_cmd(["tasklist", "/FI", f"PID eq {pid}"], timeout=10)
        return str(pid) in out
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    try:  # zombie check
        with open(f"/proc/{pid}/stat") as f:
            return f.read().split()[2] != "Z"
    except Exception:
        return True


def _kill_tree(pid: int, sig):
    if IS_WIN:
        run_cmd(["taskkill", "/T", "/F", "/PID", str(pid)], timeout=20)
        return
    try:
        os.killpg(os.getpgid(pid), sig)
    except (ProcessLookupError, PermissionError, OSError):
        try:
            os.kill(pid, sig)
        except OSError:
            pass


def stop(cfg: dict | None = None, quiet: bool = False, wait_gpu: bool = True) -> dict:
    cfg = cfg or load_config()
    state = read_json(STATE_FILE) or {}
    pids = [p["pid"] for p in state.get("procs", [])]
    for pid in pids:
        _kill_tree(pid, signal.SIGTERM)
    t0 = time.time()
    while any(_pid_alive(p) for p in pids) and time.time() - t0 < 20:
        time.sleep(0.5)
    for pid in pids:
        if _pid_alive(pid):
            _kill_tree(pid, signal.SIGKILL if not IS_WIN else signal.SIGTERM)
    if not IS_WIN:  # orphans from earlier crashed sessions / mp workers
        # never match the runner itself: only server entrypoints and the venv's mp workers
        for pat in ("vllm.entrypoints.openai.api_server", "scripts/proxy.py", "mock_vllm_server.py",
                    f"{VENV}/bin/python.*spawn_main", f"{VENV}/bin/python.*resource_tracker"):
            run_cmd(["pkill", "-9", "-f", pat], timeout=10)
    freed = None
    if wait_gpu and not state.get("mock") and gpu_query():
        # idle baseline recorded by env_check (other tenants of the GPU, e.g. a desktop) + slack
        base = read_json(ENV_DIR / "env_check.json", {}).get("gpus", [])
        thresh = {g["index"]: max(600.0, g["mem_used_mib"] + 300) for g in base}
        deadline = time.time() + cfg["timeouts"]["stop_s"]
        while time.time() < deadline:
            g = gpu_query()
            if all(x["mem_used_mib"] < thresh.get(x["index"], 600.0) for x in g):
                freed = True
                break
            time.sleep(2)
        else:
            freed = False
    if state:
        state["status"] = "stopped"
        state["stopped"] = now_iso()
        write_json(STATE_FILE, state)
    res = {"killed_pids": pids, "gpu_freed": freed, "gpu": gpu_query()}
    if not quiet:
        print(json.dumps(res, indent=2))
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("action", choices=["start", "stop", "status"])
    ap.add_argument("--arch", choices=["A", "B", "C", "D"])
    ap.add_argument("--mock", action="store_true")
    args = ap.parse_args()
    cfg = load_config()
    if args.action == "start":
        st = start(args.arch, cfg, mock=args.mock)
        print(json.dumps({k: st[k] for k in ("arch", "endpoint", "startup_s")}, indent=2))
    elif args.action == "stop":
        stop(cfg)
    else:
        st = read_json(STATE_FILE) or {}
        print(json.dumps({"arch": st.get("arch"), "status": st.get("status"), "alive": alive(st),
                          "gpu": gpu_query(), "apps": gpu_compute_apps()}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/smoke_test.py
"""Smoke test + authenticity checks for one architecture (start -> verify -> stop).

  python scripts/smoke_test.py --arch A      (P0)
  python scripts/smoke_test.py --arch C      (P1)  verifies real 2-stage pipeline
  python scripts/smoke_test.py --arch D      (P2)  verifies real KV transfer + negative control
  python scripts/smoke_test.py --arch B      (P3)  verifies two independent replicas

Writes results/smoke/<arch>.json ; the benchmark runner only runs architectures whose
smoke status is PASS (unless --ignore-smoke).
"""
from __future__ import annotations

import argparse
import asyncio
import json
import sys
import time
import traceback

import servers
from client import one_request, run_closed_loop
from common import (ARCH_LEVEL, ARCH_NAMES, SMOKE_DIR, WORKLOAD_PATH, Budget, BudgetExceeded, ensure_dirs,
                    gpu_compute_apps, gpu_query, load_config, now_iso, read_json, write_json)
from evidence import log_offsets, pd_probe_stats, proxy_stats, scan_server_logs


def common_prefix_len(a: str, b: str) -> int:
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


def check_arch(arch: str, ev: dict, gpu: list[dict], apps: list[dict], extra: dict, mock: bool) -> list[str]:
    """Return a list of failed checks (empty == pass)."""
    fails = []
    mem = {g["index"]: g["mem_used_mib"] for g in gpu}
    inst = {k: v for k, v in ev.items() if k != "proxy"}
    if not mock:
        for name, e in inst.items():
            if not (e["engine"] or "").startswith("V0"):
                fails.append(f"{name}: expected V0 engine, log says {e['engine']}")
    if arch == "A":
        e = inst["gpu0"]
        if not mock and e["pp"] != 1:
            fails.append(f"pipeline_parallel_size={e['pp']} (expected 1)")
        if not mock and not (mem.get(0, 0) > 4000 and mem.get(1, 0) < 1000):
            fails.append(f"GPU memory pattern {mem} (expected GPU0 busy, GPU1 idle)")
    elif arch == "B":
        for n in ("gpu0", "gpu1"):
            if not mock and inst[n]["pp"] != 1:
                fails.append(f"{n}: pp={inst[n]['pp']}")
            if len(inst[n]["model_loading_gib"]) != 1:
                fails.append(f"{n}: expected one full weight load, got {inst[n]['model_loading_gib']}")
        if not mock and not (mem.get(0, 0) > 4000 and mem.get(1, 0) > 4000):
            fails.append(f"GPU memory {mem}: both replicas should hold a full model + KV cache")
        pb = extra.get("proxy", {}).get("per_backend", {})
        if len(pb) < 2:
            fails.append(f"round-robin did not use both replicas: {pb}")
    elif arch == "C":
        e = inst["pp2"]
        loads = e["model_loading_gib"]
        if not mock:
            if e["pp"] != 2 or e["tp"] != 1:
                fails.append(f"tp={e['tp']} pp={e['pp']} (expected tp=1 pp=2)")
            if len(loads) != 2:
                fails.append(f"expected 2 per-stage weight loads, got {loads}")
            ref = read_json(SMOKE_DIR / "A.json") or {}
            full = (ref.get("log_evidence", {}).get("gpu0", {}).get("model_loading_gib") or [None])[0]
            extra["single_gpu_weight_gib"] = full
            # each stage holds ~half the layers (+ the tied embedding / lm_head copy), so clearly < full model
            if full and loads and max(loads) > 0.8 * full:
                fails.append(f"stage weights {loads} GiB not clearly smaller than single-GPU {full} GiB")
            if not (mem.get(0, 0) > 3000 and mem.get(1, 0) > 3000):
                fails.append(f"GPU memory {mem}: both stages should hold weights + KV cache")
        extra["pp_stage_weight_gib"] = loads
    elif arch == "D":
        pre, dec = inst["prefill"], inst["decode"]
        if not mock and not (pre["pynccl_kv_init"] and dec["pynccl_kv_init"]):
            fails.append("PyNcclConnector init line missing in a log")
        pd = extra["pd_proxied"]
        n = extra["n_proxied_requests"]
        if pd["n_select_hit"] < n:
            fails.append(f"KV hits {pd['n_select_hit']} < proxied requests {n} (decode recomputed prefill?)")
        if pd["n_recv_bypass_false"] != 0:
            fails.append(f"{pd['n_recv_bypass_false']} decode prefill steps were recomputed instead of using received KV")
        if pd["kv_bytes_total"] <= 0:
            fails.append("no KV bytes received")
        ctrl = extra["pd_control"]
        if ctrl["n_select_miss"] < 1 or ctrl["n_recv_bypass_false"] < 1:
            fails.append(f"negative control did not register a miss/recompute: {ctrl}")
    return fails


async def smoke_requests(endpoint: str, model: str, items: list[dict], timeout: float):
    seq = []
    for it in items:  # sequential: clean per-request latency
        seq.append(await run_one(endpoint, model, it, timeout))
    conc, wall, _, _ = await run_closed_loop(endpoint, model, items, len(items), timeout)
    return seq, conc


async def run_one(endpoint, model, it, timeout):
    import aiohttp
    async with aiohttp.ClientSession() as s:
        return await one_request(s, endpoint, model, it, timeout)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", choices=["A", "B", "C", "D"], required=True)
    ap.add_argument("--mock", action="store_true")
    ap.add_argument("--keep", action="store_true", help="leave servers running afterwards")
    ap.add_argument("--force", action="store_true", help="ignore level time cap")
    args = ap.parse_args()
    ensure_dirs()
    cfg = load_config()
    level = ARCH_LEVEL[args.arch]
    budget = Budget(cfg)
    try:
        budget.enter_level(level, force=args.force)
    except BudgetExceeded as e:
        print("BUDGET STOP:", e)
        sys.exit(2)
    wl = read_json(WORKLOAD_PATH)
    if not wl:
        sys.exit("workload missing: run prepare_workload.py first")
    items = wl["smoke"]
    model = cfg["model"]["served_name"]
    timeout = cfg["timeouts"]["request_s"]
    out = {"arch": args.arch, "arch_name": ARCH_NAMES[args.arch], "level": level, "time": now_iso(), "mock": args.mock,
           "status": "FAIL"}
    t_start = time.time()
    try:
        st = servers.start(args.arch, cfg, mock=args.mock, consumer_debug=(args.arch == "D"))
        out["startup_s"] = st["startup_s"]
        out["gpu_after_start"] = st["gpu_after_start"]
        out["gpu_apps_after_start"] = st["gpu_apps_after_start"]
        offs = {p["name"]: p["log_offset"] for p in st["procs"]}
        t0 = time.time()
        seq, conc = asyncio.run(smoke_requests(st["endpoint"], model, items, timeout))
        t1 = time.time()
        out["requests_sequential"] = seq
        out["requests_concurrent"] = conc
        extra = {}
        procs = {p["name"]: p for p in st["procs"]}
        if "proxy" in procs:
            extra["proxy"] = proxy_stats(procs["proxy"]["extra"].get("proxy_log"), t0, t1)
        if args.arch == "D":
            prod_log = procs["prefill"]["extra"]["probe_log"]
            cons_log = procs["decode"]["extra"]["probe_log"]
            extra["n_proxied_requests"] = len(seq) + len(conc)
            extra["pd_proxied"] = pd_probe_stats(prod_log, cons_log, t0, t1 + 1)
            # negative control: bypass the proxy -> the producer never saw this prompt
            ctrl_item = dict(items[-1])
            ctrl_item["id"] = "control-no-prefill"
            ctrl_item["prompt_token_ids"] = list(reversed(ctrl_item["prompt_token_ids"]))
            t2 = time.time()
            decode_url = f"http://127.0.0.1:{cfg['ports']['inst1']}"
            out["control_request"] = asyncio.run(run_one(decode_url, model, ctrl_item, timeout))
            time.sleep(0.5)
            extra["pd_control"] = pd_probe_stats(prod_log, cons_log, t2, time.time())
        time.sleep(1.0)
        ev = scan_server_logs(st, offs)
        out["log_evidence"] = ev
        out["gpu_after_requests"] = gpu_query()
        out["gpu_apps"] = gpu_compute_apps()
        out["extra"] = extra
        all_req = seq + conc
        bad = [r for r in all_req if not r["ok"]]
        fails = [f"request {r['id']} failed: {r.get('error')}" for r in bad]
        fails += check_arch(args.arch, ev, out["gpu_after_start"], out["gpu_apps"], extra, args.mock)
        # greedy determinism inside this architecture and agreement with A (informational)
        same = sum(1 for a, b in zip(seq, conc) if a.get("text") == b.get("text"))
        out["seq_vs_conc_identical_outputs"] = f"{same}/{len(seq)}"
        ref = read_json(SMOKE_DIR / "A.json")
        if ref and args.arch != "A" and ref.get("status") == "PASS":
            ref_t = {r["id"]: r.get("text", "") for r in ref["requests_sequential"]}
            agree = []
            for r in seq:
                a, b = ref_t.get(r["id"], ""), r.get("text", "")
                agree.append({"id": r["id"], "identical": a == b,
                              "common_prefix_chars": common_prefix_len(a, b), "len": len(b)})
            out["agreement_with_A"] = agree
        out["checks_failed"] = fails
        out["status"] = "PASS" if not fails else "FAIL"
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
        out["traceback"] = traceback.format_exc()[-4000:]
    finally:
        out["duration_s"] = round(time.time() - t_start, 1)
        if not args.keep or out["status"] != "PASS":
            out["stop"] = servers.stop(cfg, quiet=True)
        write_json(SMOKE_DIR / f"{args.arch}.json", out)
        budget.mark_level(level, "pass" if out["status"] == "PASS" else "fail", out.get("error", "")[:200])

    print(f"=== SMOKE {args.arch} ({ARCH_NAMES[args.arch]}): {out['status']}  [{out['duration_s']} s, startup {out.get('startup_s')} s]")
    for r in out.get("requests_sequential", []):
        print(f"  {r['id']:12s} ok={r['ok']} ttft={r.get('ttft_ms')} ms e2e={r.get('e2e_ms')} ms "
              f"in={r.get('prompt_tokens')} out={r.get('output_tokens')} text={r.get('text_head', '')[:60]!r}")
    for name, e in (out.get("log_evidence") or {}).items():
        print(f"  [{name}] engine={e['engine']} tp={e['tp']} pp={e['pp']} weights_GiB={e['model_loading_gib']} "
              f"attn={e['attention_backends']} kv_init={e['pynccl_kv_init']} kv_fail={e['kv_fail_count']} kv_ok={e['kv_success_count']}")
    for g in out.get("gpu_after_start", []):
        print(f"  GPU{g['index']}: {g['mem_used_mib']:.0f} MiB used after start")
    if args.arch == "D" and "extra" in out:
        p = out["extra"]["pd_proxied"]
        print(f"  PD transfer: hits={p['n_select_hit']}/{out['extra']['n_proxied_requests']} bypass_false={p['n_recv_bypass_false']} "
              f"KV bytes/req={p['kv_bytes_per_req_mean']} bytes/token={p['kv_bytes_per_token']} transfer p50={p['transfer_ms_p50']} ms "
              f"~{p['effective_GBps']} GB/s")
        print(f"  negative control (direct to decode): {out['extra']['pd_control']}")
    if out.get("agreement_with_A"):
        print("  agreement with A:", [(a["id"], a["identical"]) for a in out["agreement_with_A"]])
    for f in out.get("checks_failed", []):
        print("  CHECK FAILED:", f)
    if out.get("error"):
        print("  ERROR:", out["error"])
        print(out.get("traceback", "")[-2500:])
    print(f"  details: {SMOKE_DIR / (args.arch + '.json')}")
    sys.exit(0 if out["status"] == "PASS" else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/run_experiments.py
"""Small closed-loop benchmark: archs x {prefill_heavy, decode_heavy} x concurrency {2, 8} x reps.

  python scripts/run_experiments.py --archs A,C,D,B --reps 1        # first full round (16 runs)
  python scripts/run_experiments.py --archs A,C,D,B --reps 2        # adds rep 2 only (rep 1 is skipped)

Resumable: each run is saved atomically to results/runs/<arch>__<workload>__c<conc>__r<rep>.json
and skipped next time if it is complete and valid. Each architecture is started once,
all its runs are executed, then it is stopped and GPU memory release is verified.
"""
from __future__ import annotations

import argparse
import asyncio
import sys
import time
import traceback

import servers
from client import run_closed_loop, summarize
from common import (ARCH_NAMES, LOG_DIR, RUNS_DIR, SMOKE_DIR, WORKLOAD_PATH, Budget, BudgetExceeded, GpuSampler,
                    append_jsonl, ensure_dirs, load_config, now_iso, read_json, write_json)
from evidence import log_offsets, pd_probe_stats, proxy_stats, scan_server_logs


def run_path(arch, wname, conc, rep):
    return RUNS_DIR / f"{arch}__{wname}__c{conc}__r{rep}.json"


def is_done(path, mock: bool) -> bool:
    d = read_json(path)
    return bool(d and d.get("complete") and d.get("mock", False) == mock and d["summary"]["success_rate"] >= 0.9)


def one_run(arch, st, cfg, wl, wname, conc, rep, mock) -> dict:
    key = f"{wname}|c{conc}|r{rep}"
    entry = wl["runs"][key]
    model = cfg["model"]["served_name"]
    tmo = cfg["timeouts"]["request_s"]
    endpoint = st["endpoint"]
    # warmup at the same concurrency, discarded
    w_res, w_wall, _, _ = asyncio.run(asyncio.wait_for(
        run_closed_loop(endpoint, model, entry["warmup"], conc, tmo), cfg["timeouts"]["run_s"]))
    offs = log_offsets(st)
    with GpuSampler(cfg["gpu_sample_interval_s"]) as gs:
        res, wall, t0, t1 = asyncio.run(asyncio.wait_for(
            run_closed_loop(endpoint, model, entry["measure"], conc, tmo), cfg["timeouts"]["run_s"]))
    time.sleep(0.5)
    procs = {p["name"]: p for p in st["procs"]}
    out = {
        "arch": arch, "arch_name": ARCH_NAMES[arch], "workload": wname, "concurrency": conc, "rep": rep,
        "mock": mock, "time": now_iso(), "t_start_epoch": t0, "t_end_epoch": t1,
        "workload_sha256": wl["meta"]["runs_sha256"],
        "input_tokens": cfg["workloads"][wname]["input_tokens"], "output_tokens": cfg["workloads"][wname]["output_tokens"],
        "warmup": {"n": len(w_res), "n_ok": sum(r["ok"] for r in w_res), "wall_s": round(w_wall, 2)},
        "summary": summarize(res, wall), "gpu": gs.summary(),
        "server_log_evidence": scan_server_logs(st, offs),
        "requests": [{k: v for k, v in r.items() if k != "text"} for r in res],
    }
    if "proxy" in procs:
        out["proxy"] = proxy_stats(procs["proxy"]["extra"].get("proxy_log"), t0, t1 + 1)
    if arch == "D":
        out["pd"] = pd_probe_stats(procs["prefill"]["extra"]["probe_log"], procs["decode"]["extra"]["probe_log"], t0, t1 + 1)
    out["complete"] = True
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--archs", default="A,C,D,B")
    ap.add_argument("--workloads", default="prefill_heavy,decode_heavy")
    ap.add_argument("--conc", default=None, help="comma list, default from config")
    ap.add_argument("--reps", type=int, default=1, help="run reps 1..N (completed reps are skipped)")
    ap.add_argument("--mock", action="store_true")
    ap.add_argument("--ignore-smoke", action="store_true")
    ap.add_argument("--force", action="store_true", help="ignore the total session-time cap")
    args = ap.parse_args()
    ensure_dirs()
    cfg = load_config()
    wl = read_json(WORKLOAD_PATH)
    if not wl:
        sys.exit("workload missing: run prepare_workload.py first")
    budget = Budget(cfg)
    archs = [a.strip() for a in args.archs.split(",") if a.strip()]
    wnames = [w.strip() for w in args.workloads.split(",")]
    concs = [int(c) for c in args.conc.split(",")] if args.conc else cfg["concurrency_levels"]
    reps = list(range(1, min(args.reps, cfg["max_reps"]) + 1))
    fail_log = RUNS_DIR / "_failures.jsonl"
    summary_rows = []

    for arch in archs:
        smoke = read_json(SMOKE_DIR / f"{arch}.json")
        if not args.ignore_smoke and not (smoke and smoke.get("status") == "PASS" and smoke.get("mock", False) == args.mock):
            print(f"[{arch}] SKIP: smoke test not PASS (run smoke_test.py --arch {arch} first)")
            summary_rows.append((arch, "skipped: no smoke PASS"))
            continue
        todo = [(w, c, r) for w in wnames for c in concs for r in reps if not is_done(run_path(arch, w, c, r), args.mock)]
        if not todo:
            print(f"[{arch}] all {len(wnames) * len(concs) * len(reps)} runs already complete - skip")
            summary_rows.append((arch, "complete (cached)"))
            continue
        try:
            budget.check_total(args.force)
        except BudgetExceeded as e:
            print("BUDGET STOP:", e)
            break
        print(f"[{arch}] {ARCH_NAMES[arch]}: starting servers for {len(todo)} runs", flush=True)
        try:
            st = servers.start(arch, cfg, mock=args.mock)
        except Exception as e:
            print(f"[{arch}] server start FAILED: {e}")
            append_jsonl(fail_log, {"time": now_iso(), "arch": arch, "stage": "start", "error": str(e)[:3000]})
            summary_rows.append((arch, "server start failed"))
            continue
        st_info = {"startup_s": st["startup_s"], "gpu_after_start": st["gpu_after_start"]}
        write_json(LOG_DIR.parent / "state" / f"startup_{arch}.json", st_info)
        n_ok = 0
        try:
            for (w, c, r) in todo:
                try:
                    budget.check_total(args.force)
                except BudgetExceeded as e:
                    print("BUDGET STOP:", e)
                    break
                if not servers.alive(st):
                    raise RuntimeError("a server process died; see logs in results/logs")
                t = time.time()
                print(f"[{arch}] {w} c={c} rep={r} ...", end=" ", flush=True)
                try:
                    out = one_run(arch, st, cfg, wl, w, c, r, args.mock)
                    out["server_startup_s"] = st["startup_s"]
                    write_json(run_path(arch, w, c, r), out)
                    s = out["summary"]
                    n_ok += 1
                    msg = (f"ok {s['n_ok']}/{s['n_requests']}  TTFT p50/p95 {s['ttft_p50_ms']}/{s['ttft_p95_ms']} ms  "
                           f"ITL p50/p95 {s['itl_p50_ms']}/{s['itl_p95_ms']} ms  TPOT {s['tpot_mean_ms']} ms  "
                           f"{s['output_tok_per_s']} tok/s  ({time.time() - t:.0f}s)")
                    if arch == "D":
                        p = out["pd"]
                        msg += f"  KV hits {p['n_select_hit']} bypass_false {p['n_recv_bypass_false']} xfer p50 {p['transfer_ms_p50']} ms"
                    print(msg, flush=True)
                    if s["success_rate"] < 0.9:
                        print(f"   WARNING: success rate {s['success_rate']}, errors: {s['errors']}")
                except Exception as e:
                    print(f"FAILED: {type(e).__name__}: {e}", flush=True)
                    append_jsonl(fail_log, {"time": now_iso(), "arch": arch, "workload": w, "conc": c, "rep": r,
                                            "error": f"{type(e).__name__}: {e}"[:2000],
                                            "traceback": traceback.format_exc()[-3000:]})
            summary_rows.append((arch, f"{n_ok}/{len(todo)} new runs ok"))
        except Exception as e:
            print(f"[{arch}] ABORTED: {e}")
            append_jsonl(fail_log, {"time": now_iso(), "arch": arch, "stage": "run", "error": str(e)[:2000]})
            summary_rows.append((arch, f"aborted: {e}"[:120]))
        finally:
            res = servers.stop(cfg, quiet=True)
            print(f"[{arch}] servers stopped, GPU memory released: {res['gpu_freed']}", flush=True)

    print("\n=== run summary ===")
    for a, s in summary_rows:
        print(f"  {a}: {s}")
    print(budget.report())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/analyze_results.py
"""Aggregate runs -> summary.csv, figures/*.png, EXPERIMENT_REPORT.md, results ZIP.

Runs with the Kaggle system python (needs matplotlib only). Every number in the report
comes from results/runs/*.json and results/smoke/*.json; unmeasured cells print N/A.
"""
from __future__ import annotations

import argparse
import csv
import glob
import json
import os
import time
import zipfile
from pathlib import Path

from common import (ARCH_NAMES, ENV_DIR, RESULTS, RUNS_DIR, SMOKE_DIR, STATE_DIR, load_config, mean, percentile,
                    read_json, read_jsonl)

ARCHS = ["A", "B", "C", "D"]
ARCH_SHORT = {"A": "A 单卡混合", "B": "B DP2", "C": "C PP2", "D": "D PD分离"}
ARCH_EN = {"A": "A single", "B": "B DP2", "C": "C PP2", "D": "D PD-disagg"}
# reference categorical palette (dataviz skill), fixed order, slots 1-4
COLORS = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a", "D": "#eda100"}
WL_CN = {"prefill_heavy": "Prefill-heavy", "decode_heavy": "Decode-heavy"}


def fmt(v, nd=1):
    if v is None:
        return "N/A"
    if isinstance(v, float):
        return f"{v:.{nd}f}"
    return str(v)


def load_runs(include_mock: bool):
    runs = []
    for p in sorted(glob.glob(str(RUNS_DIR / "*__*__c*__r*.json"))):
        d = read_json(p)
        if d and d.get("complete") and (include_mock or not d.get("mock")):
            runs.append(d)
    return runs


def slo_thresholds(runs, cfg):
    s = cfg["slo"]
    out = {}
    lo = min(cfg["concurrency_levels"])
    for w in cfg["workloads"]:
        base = [r for run in runs if run["arch"] == "A" and run["workload"] == w and run["concurrency"] == lo
                for r in run["requests"] if r["ok"]]
        if base:
            out[w] = {"ttft_ms": round(s["ttft_factor"] * percentile([r["ttft_ms"] for r in base], 50), 1),
                      "tpot_ms": round(s["tpot_factor"] * percentile([r["tpot_ms"] for r in base], 50), 2),
                      "source": f"A c={lo}: {s['ttft_factor']}xTTFT_P50, {s['tpot_factor']}xTPOT_P50"}
        else:
            out[w] = dict(s["fallback_abs"][w], source="absolute fallback (A missing)")
    return out


def add_goodput(run, slo):
    t = slo[run["workload"]]
    reqs = run["requests"]
    good = [r for r in reqs if r["ok"] and r["ttft_ms"] <= t["ttft_ms"] and (r["tpot_ms"] or 0) <= t["tpot_ms"]]
    run["summary"]["slo_attainment"] = round(len(good) / len(reqs), 4) if reqs else None
    run["summary"]["goodput_req_s"] = round(len(good) / run["summary"]["wall_s"], 4)


def pd_breakdown(run):
    """Per-request join of client TTFT with proxy timings (by request id)."""
    proxy_log = RESULTS / "logs" / "proxy_D.jsonl"
    if not proxy_log.exists():
        return None
    recs = {r["rid"]: r for r in read_jsonl(proxy_log)
            if run["t_start_epoch"] - 1 <= r.get("ts", 0) <= run["t_end_epoch"] + 1}
    rows = []
    for r in run["requests"]:
        pr = recs.get(r["id"])
        if r["ok"] and pr and pr.get("prefill_ms") is not None and pr.get("decode_first_byte_ms") is not None:
            rows.append((r["ttft_ms"], pr["prefill_ms"], pr["decode_first_byte_ms"]))
    if not rows:
        return None
    xfer = (run.get("pd") or {}).get("transfer_ms_p50") or 0.0
    ttft = percentile([x[0] for x in rows], 50)
    pre = percentile([x[1] for x in rows], 50)
    dfb = percentile([x[2] for x in rows], 50)
    return {"n": len(rows), "ttft_p50": ttft, "prefill_rt_p50": pre, "decode_first_p50": dfb,
            "kv_transfer_p50": xfer, "decode_first_minus_xfer_p50": max(dfb - xfer, 0.0),
            "proxy_client_overhead_p50": percentile([x[0] - x[1] - x[2] for x in rows], 50),
            "kv_share_of_ttft": xfer / ttft if ttft else None}


def aggregate(runs):
    """(arch, workload, conc) -> mean over reps of each summary metric (+ min/max)."""
    groups = {}
    for r in runs:
        groups.setdefault((r["arch"], r["workload"], r["concurrency"]), []).append(r)
    keys = ["ttft_p50_ms", "ttft_p95_ms", "itl_p50_ms", "itl_p95_ms", "itl_p99_ms", "tpot_mean_ms", "tpot_p95_ms",
            "e2e_p50_ms", "e2e_p95_ms", "output_tok_per_s", "req_per_s", "success_rate", "slo_attainment",
            "goodput_req_s", "chunks_eq_tokens_frac"]
    agg = {}
    for k, rs in groups.items():
        a = {"n_reps": len(rs), "n_requests": sum(r["summary"]["n_requests"] for r in rs)}
        for m in keys:
            vals = [r["summary"].get(m) for r in rs if r["summary"].get(m) is not None]
            a[m] = mean(vals)
            a[m + "_min"], a[m + "_max"] = (min(vals), max(vals)) if vals else (None, None)
        g = {}
        for r in rs:
            for gi, gv in (r.get("gpu", {}).get("per_gpu") or {}).items():
                g.setdefault(gi, {"mem": [], "util": []})
                g[gi]["mem"].append(gv["mem_peak_mib"])
                g[gi]["util"].append(gv["util_mean_pct"])
        a["gpu"] = {gi: {"mem_peak_mib": max(v["mem"]), "util_mean_pct": mean(v["util"])} for gi, v in g.items()}
        if k[0] == "D":
            pds = [r.get("pd") for r in rs if r.get("pd")]
            a["pd"] = {f: mean([p.get(f) for p in pds]) for f in ("transfer_ms_p50", "transfer_ms_p95", "kv_bytes_per_req_mean",
                                                                    "effective_GBps", "kv_bytes_per_token")}
            a["pd"]["n_select_hit"] = sum(p["n_select_hit"] for p in pds)
            a["pd"]["n_recv_bypass_false"] = sum(p["n_recv_bypass_false"] for p in pds)
            bds = [b for b in (pd_breakdown(r) for r in rs) if b]
            if bds:
                a["pd_breakdown"] = {f: mean([b[f] for b in bds if b[f] is not None]) for f in bds[0] if f != "n"}
        agg[k] = a
    return agg


# ---------------------------------------------------------------- figures
def figures(agg, cfg, outdir: Path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as e:
        print("matplotlib unavailable, skipping figures:", e)
        return []
    outdir.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                         "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                         "ytick.color": "#52514e", "axes.titlecolor": "#0b0b0b", "axes.grid": True,
                         "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True})
    groups = [(w, c) for w in cfg["workloads"] for c in cfg["concurrency_levels"]]
    made = []

    def grouped(metric, title, ylabel, fname, digits=0):
        fig, ax = plt.subplots(figsize=(8, 3.6), dpi=140)
        width = 0.19
        for i, a in enumerate(ARCHS):
            xs, ys = [], []
            for gi, (w, c) in enumerate(groups):
                v = agg.get((a, w, c), {}).get(metric)
                if v is not None:
                    xs.append(gi + (i - 1.5) * (width + 0.01))
                    ys.append(v)
            if xs:
                bars = ax.bar(xs, ys, width=width, color=COLORS[a], label=ARCH_EN[a], edgecolor="#fcfcfb", linewidth=1)
                for x, y in zip(xs, ys):
                    ax.text(x, y, f"{y:.{digits}f}", ha="center", va="bottom", fontsize=6.5, color="#52514e")
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{WL_CN[w]}\nconc={c}" for w, c in groups])
        ax.set_ylabel(ylabel)
        ax.set_title(title, loc="left", fontsize=10)
        ax.legend(ncol=4, frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0, -0.18))
        fig.tight_layout()
        p = outdir / fname
        fig.savefig(p)
        plt.close(fig)
        made.append(p)

    grouped("ttft_p95_ms", "TTFT P95 (ms, lower is better)", "ms", "ttft_p95.png")
    grouped("itl_p95_ms", "ITL P95 per streamed token (ms, lower is better)", "ms", "itl_p95.png", 1)
    grouped("output_tok_per_s", "Output throughput (tokens/s, higher is better)", "tok/s", "throughput.png")
    grouped("tpot_mean_ms", "TPOT mean (ms/token, lower is better)", "ms", "tpot_mean.png", 1)
    grouped("goodput_req_s", "Goodput (req/s meeting TTFT+TPOT SLO)", "req/s", "goodput.png", 2)

    # PD breakdown
    rows = [((w, c), agg[("D", w, c)]["pd_breakdown"]) for w, c in groups if ("D", w, c) in agg and agg[("D", w, c)].get("pd_breakdown")]
    if rows:
        fig, ax = plt.subplots(figsize=(8, 3.2), dpi=140)
        parts = [("prefill_rt_p50", "prefill request round-trip (queue+prefill on GPU0)", "#2a78d6"),
                 ("kv_transfer_p50", "KV transfer (PyNccl select)", "#eb6834"),
                 ("decode_first_minus_xfer_p50", "decode first token (excl. transfer)", "#1baf7a"),
                 ("proxy_client_overhead_p50", "proxy/client overhead", "#8a8984")]
        for yi, (g, b) in enumerate(rows):
            left = 0
            for k, lab, col in parts:
                v = max(b.get(k) or 0, 0)
                ax.barh(yi, v, left=left, color=col, edgecolor="#fcfcfb", linewidth=1.5, height=0.55,
                        label=lab if yi == 0 else None)
                left += v
            ax.text(left, yi, f"  TTFT p50 {b['ttft_p50']:.0f} ms", va="center", fontsize=7, color="#52514e")
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels([f"{WL_CN[w]} c={c}" for (w, c), _ in rows])
        ax.set_xlabel("ms (median of each component; components are approximate)")
        ax.set_title("D: PD-disaggregated TTFT decomposition", loc="left", fontsize=10)
        ax.legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(0, -0.25), ncol=2)
        fig.tight_layout()
        p = outdir / "pd_ttft_breakdown.png"
        fig.savefig(p)
        plt.close(fig)
        made.append(p)
    return made


# ---------------------------------------------------------------- report
def pctf(v):
    return "N/A" if v is None else f"{v:.0f}%"


def ratio(a, b):
    return a / b if (a is not None and b not in (None, 0)) else None


def pct_change(a, b):
    r = ratio(a, b)
    return None if r is None else (r - 1) * 100


def observations(agg, cfg):
    """Purely data-driven sentences; nothing here is asserted without the numbers."""
    obs = []
    lo, hi = min(cfg["concurrency_levels"]), max(cfg["concurrency_levels"])
    for w in cfg["workloads"]:
        for c in cfg["concurrency_levels"]:
            present = [a for a in ARCHS if (a, w, c) in agg]
            if len(present) < 2:
                continue
            for m, better, name in (("ttft_p95_ms", min, "TTFT P95"), ("itl_p95_ms", min, "ITL P95"),
                                    ("output_tok_per_s", max, "输出吞吐"), ("goodput_req_s", max, "Goodput")):
                vals = {a: agg[(a, w, c)][m] for a in present if agg[(a, w, c)].get(m) is not None}
                if vals and not any(vals.values()):
                    obs.append(f"{WL_CN[w]} c={c}：{name} 全部为 0（没有架构满足 SLO）")
                elif vals:
                    best = better(vals, key=vals.get)
                    obs.append(f"{WL_CN[w]} c={c}：{name} 最优为 {ARCH_SHORT[best]}（{fmt(vals[best])}）；"
                               + "，".join(f"{ARCH_SHORT[a]}={fmt(v)}" for a, v in vals.items()))
    return obs


def comparison_lines(agg, cfg):
    """Key ratios used by the mechanism discussion."""
    L = {}
    lo, hi = min(cfg["concurrency_levels"]), max(cfg["concurrency_levels"])
    for w in cfg["workloads"]:
        for c in cfg["concurrency_levels"]:
            A = agg.get(("A", w, c))
            for x in "BCD":
                X = agg.get((x, w, c))
                if A and X:
                    L[(x, w, c)] = {m: pct_change(X.get(m), A.get(m)) for m in
                                    ("ttft_p50_ms", "ttft_p95_ms", "itl_p50_ms", "itl_p95_ms", "tpot_mean_ms",
                                     "output_tok_per_s", "goodput_req_s")}
    return L


def sgn(v, unit="%"):
    return "N/A" if v is None else f"{v:+.0f}{unit}"


def mechanism_text(agg, cfg, cmp):
    lo, hi = min(cfg["concurrency_levels"]), max(cfg["concurrency_levels"])
    T = []

    def get(x, w, c, m):
        return (cmp.get((x, w, c)) or {}).get(m)

    # PP2
    pp = [(w, c, get("C", w, c, "ttft_p50_ms"), get("C", w, c, "tpot_mean_ms")) for w in cfg["workloads"] for c in cfg["concurrency_levels"] if ("C", w, c) in cmp]
    if pp:
        T.append("**PP2（C）相对单卡（A）**：" + "；".join(f"{WL_CN[w]} c={c} TTFT P50 {sgn(a)}、TPOT {sgn(b)}" for w, c, a, b in pp) + "。")
        slower = [x for x in pp if x[3] is not None and x[3] > 5]
        faster = [x for x in pp if x[3] is not None and x[3] < -5]
        T.append(f"TPOT 比单卡高 5% 以上的配置 {len(slower)}/{len(pp)} 个，低 5% 以上的 {len(faster)}/{len(pp)} 个。"
                 "机制上：1.5B 模型单卡即可放下，PP 不减少每个 token 的计算量（仍要串行经过全部 28 层），却在每个 step 增加一次跨卡激活传输（PCIe、无 NVLink）"
                 "以及两个 stage 之间的同步/调度开销；vLLM V0 用 2 个 virtual engine 交错不同批次来填充流水线，只有同时有 ≥2 个批次在飞时才可能摊薄气泡。"
                 + ("本次实测中 PP2 的 TPOT 高于单卡的配置占多数，与上述开销一致。" if len(slower) > len(pp) / 2 else
                    "本次实测并未出现“PP2 普遍更慢”，与上述预期不一致；需要结合两卡利用率与更多重复来判断，样本有限，不做强结论。"))
    # DP2
    dp = [(w, c, get("B", w, c, "output_tok_per_s"), get("B", w, c, "ttft_p50_ms")) for w in cfg["workloads"] for c in cfg["concurrency_levels"] if ("B", w, c) in cmp]
    if dp:
        T.append("**DP2（B）相对单卡**：" + "；".join(f"{WL_CN[w]} c={c} 输出吞吐 {sgn(v)}、TTFT P50 {sgn(t)}" for w, c, v, t in dp) + "。")
        gains = [v for _, _, v, _ in dp if v is not None]
        best = max(gains) if gains else None
        T.append("DP2 是两份完全独立的副本：单个请求不会被两张卡加速，收益只来自把并发请求分到两份计算/KV 资源上（每个副本的有效 batch 减半、prefill 排队被分摊）。"
                 f"理论吞吐上限为单卡的 2 倍（+100%），本次实测最大增益 {sgn(best)}。"
                 "单卡未饱和时（小模型、低并发，decode 以访存为主，batch 增大时步时几乎不变）增益通常明显小于 +100%；"
                 "闭环固定并发下若单卡的排队延迟随 batch 超线性增长，DP2 把每卡并发减半后增益也可能超过 +100%。低并发（c=2）下 DP2 还要承担代理转发开销。"
                 "是否“值得”两张卡，应看相同 SLO 下的 Goodput 而不是峰值吞吐：见表中 Goodput 列。")
    # PD
    pdl = [(w, c, get("D", w, c, "itl_p95_ms"), get("D", w, c, "ttft_p50_ms"), get("D", w, c, "output_tok_per_s"))
           for w in cfg["workloads"] for c in cfg["concurrency_levels"] if ("D", w, c) in cmp]
    if pdl:
        T.append("**PD 分离（D）相对单卡**：" + "；".join(f"{WL_CN[w]} c={c} ITL P95 {sgn(i)}、TTFT P50 {sgn(t)}、吞吐 {sgn(o)}" for w, c, i, t, o in pdl) + "。")
        improved = [x for x in pdl if x[2] is not None and x[2] < -5]
        worse = [x for x in pdl if x[2] is not None and x[2] > 5]
        if improved:
            T.append("在 " + "、".join(f"{WL_CN[w]} c={c}" for w, c, *_ in improved) +
                     " 中 D 的 ITL P95 比单卡低 5% 以上，与 PD 分离的核心机制一致：V0 关闭 chunked prefill 时，新请求的整段 prefill 作为独立 step 插入，"
                     "正在 decode 的请求必须等它完成，单卡混合部署的 ITL 尾部因此被拉长；D 把 prefill 移到 GPU0，GPU1 上对应的 step 只需把收到的 KV 写入分页缓存"
                     "（bypass_model_exec=True），对 decode 的打断短得多。")
        if worse:
            T.append("在 " + "、".join(f"{WL_CN[w]} c={c}" for w, c, *_ in worse) +
                     " 中 D 的 ITL P95 反而比单卡高 5% 以上：consumer 侧接收 KV（同步阻塞的 PyNccl select）和写入分页缓存的 step 同样会插在 decode step 之间，"
                     "当这部分耗时与单卡上一次 prefill 相当（输入短、prefill 本来就快）时，PD 分离无法改善甚至恶化 decode 尾延迟。")
        if not improved:
            T.append("本次数据中没有任何配置的 ITL P95 改善超过 5%，因此不能声称 PD 分离改善了 decode 尾延迟。")
        tt = [x for x in pdl if x[3] is not None]
        if tt:
            n_up = sum(1 for x in tt if x[3] > 0)
            T.append(f"TTFT：D 的 TTFT P50 高于单卡的配置 {n_up}/{len(tt)} 个。D 的首 token 路径是“代理 → prefill 实例完成 prefill（max_tokens=1）→ 返回代理 → 代理请求 decode 实例 → "
                     "decode 端经 PyNccl 拉取 KV 与 hidden states → 写入分页 KV → 采样首 token”，比单卡多一次 HTTP 往返、一次串行 KV 传输和一次 consumer 侧调度；"
                     "只有在高并发、单卡上 prefill 排队严重时，D 才可能因为 prefill 有专属 GPU 而降低 TTFT。")
        tp = [x for x in pdl if x[4] is not None]
        if tp:
            n_up = sum(1 for x in tp if x[4] > 5)
            T.append(f"吞吐：D 的输出吞吐比单卡高 5% 以上的配置 {n_up}/{len(tp)} 个。D 用两张卡，但吞吐受两端中较慢的一端限制："
                     "Decode-heavy 下 GPU0 大部分时间只做很短的 prefill，GPU1 承担全部 decode，等效于“一张卡 decode + 一张卡大部分空闲”，因此 PD 分离不一定提升总吞吐（可对照 GPU 利用率表）。")
    # KV transfer cost
    kv = [(w, c, agg[("D", w, c)]) for w in cfg["workloads"] for c in cfg["concurrency_levels"] if ("D", w, c) in agg]
    if kv:
        parts = []
        for w, c, a in kv:
            p = a.get("pd", {})
            b = a.get("pd_breakdown", {})
            share = b.get("kv_share_of_ttft")
            parts.append(f"{WL_CN[w]} c={c}：每请求 KV {fmt((p.get('kv_bytes_per_req_mean') or 0) / 1e6, 1)} MB，传输 P50 {fmt(p.get('transfer_ms_p50'))} ms"
                         f"（约 {fmt(p.get('effective_GBps'), 2)} GB/s），占 TTFT P50 约 {fmt(share * 100 if share is not None else None, 0)}%")
        T.append("**KV Cache 传输成本**：" + "；".join(parts) + "。理论值：Qwen2.5-1.5B 每 token KV = 28 层 × 2(K,V) × 2 KV 头 × 128 维 × 2 字节 = 28,672 字节，"
                 "1024 token ≈ 29.4 MB（另外 SimpleConnector 还传输全部 token 的 hidden states，约 3 KB/token）。GQA（2 个 KV 头）让这个模型的 KV 很小；"
                 "对 MHA 或更大的模型、更长的输入，传输量成比例增长，而 T4 之间只有 PCIe（Kaggle 上通常无 P2P，经主机内存中转），传输占比会显著上升，"
                 "这就是 PD 收益可能被传输成本抵消的条件：输入很长、KV 头多、链路慢，而 decode 侧被 prefill 打断的代价又不大（低并发、短输入）。")
    return T


def report(cfg, runs, agg, slo, figs, include_mock):
    env = read_json(ENV_DIR / "env_check.json", {})
    inst = read_json(ENV_DIR / "install.json", {})
    model = read_json(ENV_DIR / "model.json", {})
    budget = read_json(STATE_DIR / "budget.json", {})
    smoke = {a: read_json(SMOKE_DIR / f"{a}.json") for a in ARCHS}
    v = inst.get("verify", {})
    mock = any(r.get("mock") for r in runs) or any((s or {}).get("mock") for s in smoke.values())
    L = []
    L.append("# Kaggle 2×T4：单卡 / DP2 / PP2 / PD 分离推理架构小规模对比实验报告\n")
    if mock:
        L.append("> ⚠️ **本报告包含 MOCK（本地假服务器）数据，仅用于测试流水线，不代表任何真实性能。**\n")
    L.append(f"生成时间：{time.strftime('%Y-%m-%d %H:%M:%S')}。所有数值均由 `results/runs/*.json` 与 `results/smoke/*.json` 自动汇总；未测量项为 N/A。\n")

    L.append("## 1. 实验环境\n")
    gpus = env.get("gpus", [])
    L.append("| 项目 | 值 |\n|---|---|")
    gpu_desc = ", ".join("GPU%d %s %.0f MiB" % (g["index"], g["name"], g["mem_total_mib"]) for g in gpus) or "N/A"
    L.append(f"| GPU | {gpu_desc} |")
    L.append(f"| GPU 细节 (compute cap, driver, PCIe gen/width) | {env.get('gpu_details_raw', 'N/A').replace(chr(10), ' / ')} |")
    L.append(f"| GPU P2P (torch.can_device_access_peer 0→1) | {v.get('p2p_0_1', 'N/A')} |")
    L.append(f"| vLLM / 引擎 | {v.get('vllm', 'N/A')}，V0 引擎（VLLM_USE_V1=0；0.8.5 在 CC<8.0 上本就回退 V0），XFormers attention |")
    L.append(f"| PyTorch / CUDA / NCCL | {v.get('torch', 'N/A')} / {v.get('torch_cuda', 'N/A')} / {v.get('nccl', 'N/A')} |")
    L.append(f"| transformers / xformers / Python | {v.get('transformers', 'N/A')} / {v.get('xformers', 'N/A')} / {v.get('python', 'N/A')} |")
    L.append(f"| 依赖解析 | uv `--exclude-newer {inst.get('exclude_newer', 'N/A')}`，完整列表见 env/venv_freeze.txt |")
    L.append(f"| 模型 | {model.get('repo_id', 'N/A')} @ `{model.get('revision', 'N/A')}`，FP16（checkpoint 为 {model.get('torch_dtype_in_checkpoint', '?')}），权重 {model.get('weights_gb', 'N/A')} GB |")
    vc = cfg["vllm"]
    L.append(f"| 统一服务参数 | dtype={vc['dtype']}, max_model_len={vc['max_model_len']}, gpu_mem_util={vc['gpu_memory_utilization']}, max_num_seqs={vc['max_num_seqs']}, "
             f"max_num_batched_tokens={vc['max_num_batched_tokens']}, chunked prefill=OFF, prefix caching=OFF, CUDA graph={'OFF (eager)' if vc['enforce_eager'] else 'ON'}, seed={vc['seed']} |")
    L.append(f"| 采样 | temperature=0, ignore_eos=true, 固定 max_tokens；prompt 以 token ID 发送，所有架构完全相同 |")
    if budget:
        L.append(f"| 会话计时 | 自 {budget.get('session_start_iso')} 起，生成报告时已用 {(time.time() - budget['session_start']) / 3600:.2f} h |")
    L.append("")

    L.append("## 2. 架构对照与真实性验证\n")
    L.append("| 架构 | 启动方式 | 权重分布 | KV Cache 路径 | Smoke | 启动耗时 |\n|---|---|---|---|---|---|")
    desc = {
        "A": ("1 个 vLLM 实例，CUDA_VISIBLE_DEVICES=0", "GPU0 完整模型，GPU1 空闲", "GPU0 本地分页 KV"),
        "B": ("2 个独立 vLLM 实例 + 轮询代理", "GPU0、GPU1 各一份完整模型", "各副本本地，请求级分发"),
        "C": ("1 个 vLLM 实例，--pipeline-parallel-size 2 --tensor-parallel-size 1，mp backend", "GPU0 前 14 层+embedding，GPU1 后 14 层+lm_head", "各 stage 保存自己层的 KV；层间传激活"),
        "D": ("prefill 实例(kv_producer, GPU0) + decode 实例(kv_consumer, GPU1)，PyNcclConnector + PD 代理", "两卡各一份完整模型", "GPU0 prefill → PyNccl(NCCL) 发送 KV+hidden → GPU1 写入分页 KV 后继续 decode"),
    }
    for a in ARCHS:
        s = smoke[a] or {}
        L.append(f"| {ARCH_SHORT[a]} | {desc[a][0]} | {desc[a][1]} | {desc[a][2]} | {s.get('status', '未运行')} | {fmt(s.get('startup_s'))} s |")
    L.append("")
    L.append("**真实性证据（来自 smoke test 的日志与 nvidia-smi）：**\n")
    for a in ARCHS:
        s = smoke[a]
        if not s:
            L.append(f"- {ARCH_SHORT[a]}：未运行 smoke test。")
            continue
        ev = s.get("log_evidence") or {}
        mem = {g["index"]: round(g["mem_used_mib"]) for g in s.get("gpu_after_start", [])}
        items = [f"{n}: engine={e['engine']}, tp={e['tp']}, pp={e['pp']}, 权重加载={e['model_loading_gib']} GiB" for n, e in ev.items() if n != "proxy"]
        line = f"- {ARCH_SHORT[a]}（{s['status']}）：" + "；".join(items) + f"；启动后显存 MiB {mem}"
        if a == "B" and s.get("extra", {}).get("proxy"):
            line += f"；代理分发计数 {s['extra']['proxy'].get('per_backend')}"
        if a == "D" and s.get("extra"):
            p, c = s["extra"].get("pd_proxied", {}), s["extra"].get("pd_control", {})
            line += (f"；经代理的 {s['extra'].get('n_proxied_requests')} 个请求中 KV 命中 {p.get('n_select_hit')} 次、"
                     f"decode 端重算 prefill {p.get('n_recv_bypass_false')} 次，官方 DEBUG 日志 'Successfully received all KVs' {ev.get('decode', {}).get('kv_success_count')} 次；"
                     f"负对照（绕过 prefill 直接发给 decode）：未命中 {c.get('n_select_miss')}、重算 {c.get('n_recv_bypass_false')}，"
                     f"官方 WARNING 'Failed to receive all KVs' {ev.get('decode', {}).get('kv_fail_count')} 次（应只来自负对照）")
        if s.get("agreement_with_A"):
            same = sum(x["identical"] for x in s["agreement_with_A"])
            line += f"；与 A 的贪心输出逐字一致 {same}/{len(s['agreement_with_A'])}（FP16 下不同 kernel/批次可能导致少量差异，仅作合理性参考）"
        if s.get("checks_failed"):
            line += f"；**失败检查：{s['checks_failed']}**"
        if s.get("error"):
            line += f"；**错误：{s['error'][:300]}**"
        L.append(line)
    L.append("")

    L.append("## 3. 性能结果\n")
    L.append(f"负载：闭环固定并发（{cfg['concurrency_levels']}），每个配置 {cfg['warmup_requests']} 个预热请求（不计入）+ 正式请求；"
             f"Prefill-heavy = {cfg['workloads']['prefill_heavy']['input_tokens']} 输入 / {cfg['workloads']['prefill_heavy']['output_tokens']} 输出 token，"
             f"Decode-heavy = {cfg['workloads']['decode_heavy']['input_tokens']} / {cfg['workloads']['decode_heavy']['output_tokens']}。\n")
    L.append("指标定义：TTFT=客户端发出请求到首个携带生成 token 的流式事件；ITL=相邻 token 事件的间隔（全部请求合并后取分位数，`chunk=token` 比例见表）；"
             "TPOT=单请求 (E2E−TTFT)/(输出token−1)；Goodput=同时满足 TTFT 与 TPOT SLO 的请求数/墙钟时间。\n")
    L.append("SLO（实验前在 configs/experiments.json 中预先登记的规则）：" + "；".join(f"{WL_CN[w]} TTFT≤{t['ttft_ms']} ms、TPOT≤{t['tpot_ms']} ms（{t['source']}）" for w, t in slo.items()) + "\n")
    for w in cfg["workloads"]:
        L.append(f"### {WL_CN[w]}\n")
        L.append("| 架构 | 并发 | reps | 请求 | 成功率 | TTFT P50 | TTFT P95 | ITL P50 | ITL P95 | TPOT 均值 | E2E P95 | 输出 tok/s | req/s | SLO 达标 | Goodput req/s | chunk=token |")
        L.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|")
        for c in cfg["concurrency_levels"]:
            for a in ARCHS:
                g = agg.get((a, w, c))
                if not g:
                    L.append(f"| {ARCH_SHORT[a]} | {c} | 0 | N/A |" + " N/A |" * 12)
                    continue
                L.append(f"| {ARCH_SHORT[a]} | {c} | {g['n_reps']} | {g['n_requests']} | {fmt((g['success_rate'] or 0) * 100, 0)}% | {fmt(g['ttft_p50_ms'])} | {fmt(g['ttft_p95_ms'])} | "
                         f"{fmt(g['itl_p50_ms'])} | {fmt(g['itl_p95_ms'])} | {fmt(g['tpot_mean_ms'])} | {fmt(g['e2e_p95_ms'], 0)} | {fmt(g['output_tok_per_s'])} | "
                         f"{fmt(g['req_per_s'], 2)} | {fmt((g['slo_attainment'] or 0) * 100, 0)}% | {fmt(g['goodput_req_s'], 2)} | {fmt((g['chunks_eq_tokens_frac'] or 0) * 100, 0)}% |")
        L.append("")
    if any(g["n_reps"] > 1 for g in agg.values()):
        L.append("多次重复时表中为各 rep 指标的均值；各 rep 的最小/最大值见 summary.csv。\n")

    L.append("### GPU 资源（正式计时窗口内 nvidia-smi 采样）\n")
    L.append("| 架构 | 负载 | 并发 | GPU0 显存峰值 MiB | GPU0 平均利用率 | GPU1 显存峰值 MiB | GPU1 平均利用率 |\n|---|---|---|---|---|---|---|")
    for (a, w, c), g in sorted(agg.items()):
        g0, g1 = g["gpu"].get("0", {}), g["gpu"].get("1", {})
        L.append(f"| {ARCH_SHORT[a]} | {WL_CN[w]} | {c} | {fmt(g0.get('mem_peak_mib'), 0)} | {pctf(g0.get('util_mean_pct'))} | {fmt(g1.get('mem_peak_mib'), 0)} | {pctf(g1.get('util_mean_pct'))} |")
    L.append("\n注：显存峰值主要由 `gpu_memory_utilization` 预分配决定，不代表实际 KV 使用量；利用率为 nvidia-smi 的 kernel 活跃时间比例，粒度粗。\n")

    dkeys = [(w, c) for w in cfg["workloads"] for c in cfg["concurrency_levels"] if ("D", w, c) in agg]
    if dkeys:
        L.append("### PD 分离：KV 传输与 TTFT 分解\n")
        L.append("| 负载 | 并发 | KV 命中 | decode 端重算 | 每请求 KV MB | 字节/token | 传输 P50 ms | 传输 P95 ms | 有效带宽 GB/s | 客户端 TTFT P50 | prefill 往返 P50 | decode 首包 P50 | 代理+客户端开销 P50 |")
        L.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
        for w, c in dkeys:
            g = agg[("D", w, c)]
            p, b = g.get("pd", {}), g.get("pd_breakdown", {})
            L.append(f"| {WL_CN[w]} | {c} | {p.get('n_select_hit')} | {p.get('n_recv_bypass_false')} | {fmt((p.get('kv_bytes_per_req_mean') or 0) / 1e6, 2)} | "
                     f"{fmt(p.get('kv_bytes_per_token'), 0)} | {fmt(p.get('transfer_ms_p50'), 2)} | {fmt(p.get('transfer_ms_p95'), 2)} | {fmt(p.get('effective_GBps'), 2)} | "
                     f"{fmt(b.get('ttft_p50'))} | {fmt(b.get('prefill_rt_p50'))} | {fmt(b.get('decode_first_p50'))} | {fmt(b.get('proxy_client_overhead_p50'))} |")
        L.append("\n传输时间 = consumer 侧 `SimpleConnector.select` 的阻塞时间（含信令往返、5 个张量的元数据与 NCCL 接收，探针在其后 `cuda.synchronize`），"
                 "字节数 = 收到的 K+V 张量大小（hidden states 另计）。各分量取各自中位数，相加不严格等于 TTFT 中位数。\n")

    L.append("## 4. 图表\n")
    for f in figs:
        L.append(f"![{f.stem}](figures/{f.name})\n")

    L.append("## 5. 结果解读\n")
    L.append("### 5.1 数据直接给出的结论（自动生成）\n")
    for o in observations(agg, cfg):
        L.append(f"- {o}")
    L.append("")
    L.append("### 5.2 机制解释（措辞随实测方向自动选择；样本少，均为描述性结论）\n")
    for t in mechanism_text(agg, cfg, comparison_lines(agg, cfg)):
        L.append(t + "\n")
    L.append("**哪些差异来自硬件、哪些来自架构**：T4 FP16 算力（~65 TFLOPS 峰值）与 320 GB/s 显存带宽决定了单卡 prefill/decode 的绝对速度；"
             "PCIe 互联（无 NVLink，Kaggle 上通常无 GPU P2P）决定了 PP 激活传输与 PD KV 传输的单位成本；1.5B 小模型使单卡远未饱和，削弱了 DP/PP 的意义。"
             "架构本身的影响体现在：PP 让每个 token 串行经过两张卡；DP 只增加并发容量不缩短单请求；PD 用一次 KV 传输换取 prefill 与 decode 的隔离。\n")

    L.append("## 6. 局限性\n")
    for t in ["只有 2 张 T4（Turing，无 bf16、无 FlashAttention-2，使用 V0 + XFormers），与 H100/B200 + NVLink/RDMA 的生产集群差异巨大；",
              "模型只有 1.5B 且是 GQA（2 个 KV 头），KV 体积很小，PD 传输成本被低估，PP/DP 的收益也被低估；",
              "PD 使用的是 vLLM V0 的 PyNcclConnector/SimpleConnector 参考实现：KV 以整请求为单位、prefill 完成后同步拉取，不与计算重叠，"
              "也传输了全部 token 的 hidden states；生产实现（V1 的 NIXL/P2P NCCL/LMCache，layer-wise 异步传输）开销模型不同；",
              "关闭了 chunked prefill（SimpleConnector 的要求），统一应用到所有架构；开启 chunked prefill 的单卡混合部署会显著缓解 decode 被 prefill 打断的问题，结论不能外推；",
              "闭环负载、每配置 20–40 个请求、重复次数少，P95 的统计不确定性很大，只报告描述性统计，不声称统计显著或容量边界；",
              "Kaggle 是共享虚拟化环境，PCIe 拓扑、CPU 争用、nvidia-smi 采样粒度都会引入噪声；",
              "DP2 与 PD 经过一个 Python aiohttp 代理，A/C 由客户端直连，代理开销（localhost，通常 ~1 ms 量级）计入 B/D 的 TTFT。"]:
        L.append(f"- {t}")
    L.append("")

    L.append("## 7. 完成状态\n")
    planned = [(a, w, c) for a in ARCHS for w in cfg["workloads"] for c in cfg["concurrency_levels"]]
    done = [k for k in planned if k in agg]
    L.append(f"主矩阵完成 {len(done)}/{len(planned)} 个配置。")
    missing = [k for k in planned if k not in agg]
    if missing:
        L.append("未完成（表中为 N/A，不得视为已测量）：" + "、".join(f"{ARCH_SHORT[a]}/{WL_CN[w]}/c={c}" for a, w, c in missing))
    fails = read_jsonl(RUNS_DIR / "_failures.jsonl")
    if fails:
        L.append(f"\n运行失败记录 {len(fails)} 条（results/runs/_failures.jsonl），最近一条：`{fails[-1].get('error', '')[:300]}`")
    L.append("")

    L.append("## 8. 简历描述建议（仅基于已完成并验证的内容）\n")
    ok_arch = [a for a in ARCHS if (smoke[a] or {}).get("status") == "PASS" and any(k[0] == a for k in agg)]
    if mock:
        L.append("（MOCK 数据，不生成简历描述。）")
    elif not ok_arch:
        L.append("（尚无通过验证的真实测量，暂不生成。）")
    else:
        names = "、".join({"A": "单卡混合", "B": "DP2", "C": "PP2", "D": "PD 分离"}[a] for a in ok_arch)
        L.append(f"- 在 Kaggle 2×T4 上基于 vLLM {v.get('vllm', '')} 搭建 {names} {len(ok_arch)} 种推理部署架构的可复现对比实验（Qwen2.5-1.5B，FP16），"
                 "编写流式压测客户端统一测量 TTFT/ITL/TPOT/吞吐/Goodput，实现断点续跑、超时与 GPU 资源回收。")
        if "D" in ok_arch:
            d = [agg[k] for k in agg if k[0] == "D" and agg[k].get("pd")]
            if d:
                kvmb = mean([x["pd"].get("kv_bytes_per_req_mean") for x in d])
                xf = mean([x["pd"].get("transfer_ms_p50") for x in d])
                L.append(f"- 基于 vLLM PyNcclConnector 部署真实 Prefill/Decode 分离（GPU0 prefill → NCCL 传输 KV → GPU1 decode），通过无侵入探针统计 KV 传输量与耗时"
                         f"（实测每请求平均 {fmt((kvmb or 0) / 1e6, 1)} MB、传输中位数 {fmt(xf, 1)} ms），并用绕过 prefill 的负对照验证 decode 端未重算 prefill。")
        if "C" in ok_arch:
            L.append("- 通过日志与显存证据验证 PP2 的层切分（两个 stage 各加载约一半权重），结合 TTFT/TPOT 数据分析小模型在 PCIe 互联下流水线并行的通信与气泡开销。")
        L.append("（请把上面的数字与本报告表格核对后再使用；不要写入未在表中出现的提升百分比。）")
    L.append("")
    (RESULTS / "EXPERIMENT_REPORT.md").write_text("\n".join(L), encoding="utf-8")


def write_csv(agg, path):
    keys = sorted({k for g in agg.values() for k, v in g.items() if not isinstance(v, dict)})
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["arch", "workload", "concurrency"] + keys)
        for (a, wl, c), g in sorted(agg.items()):
            w.writerow([a, wl, c] + [g.get(k) for k in keys])


def make_zip(out_dir: Path) -> Path:
    ts = time.strftime("%Y%m%d_%H%M%S")
    zp = out_dir / f"pdpp_results_{ts}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(RESULTS):
            for fn in files:
                p = Path(root) / fn
                if p.suffix == ".zip":
                    continue
                if p.stat().st_size > 200 * 1024 * 1024:
                    continue
                z.write(p, Path("results") / p.relative_to(RESULTS))
        for extra in ("configs/experiments.json",):
            p = RESULTS.parent / extra
            if p.exists():
                z.write(p, extra)
    return zp


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--include-mock", action="store_true")
    ap.add_argument("--zip-dir", default=None, help="default: parent of the lab dir (/kaggle/working)")
    ap.add_argument("--no-zip", action="store_true")
    args = ap.parse_args()
    cfg = load_config()
    runs = load_runs(args.include_mock)
    slo = slo_thresholds(runs, cfg)
    for r in runs:
        add_goodput(r, slo)
    agg = aggregate(runs)
    write_csv(agg, RESULTS / "summary.csv")
    figs = figures(agg, cfg, RESULTS / "figures")
    report(cfg, runs, agg, slo, figs, args.include_mock)
    print(f"{len(runs)} runs, {len(agg)} configs -> {RESULTS / 'EXPERIMENT_REPORT.md'}, {len(figs)} figures")
    if not args.no_zip:
        zp = make_zip(Path(args.zip_dir) if args.zip_dir else RESULTS.parent.parent)
        print("ZIP:", zp)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/pd_probe/sitecustomize.py
"""PD KV-transfer probe, auto-imported by Python's `site` module when this directory is on
PYTHONPATH (the server launcher only adds it for architecture D).

It does NOT change vLLM behaviour: it wraps four methods of vLLM 0.8.5's
`SimpleConnector` (PyNcclConnector) and appends one JSON line per call to
$PDLAB_PROBE_LOG:

  producer:  send_batch (gather KV from paged cache + enqueue), insert (bytes enqueued)
  consumer:  select     (blocking fetch of one request's KV+hidden over PyNccl, incl. signalling)
             recv_batch (all selects + writing KV into the paged cache) with the official
                        `bypass_model_exec` flag: True means the decode instance skipped the
                        prefill forward pass and used the transferred KV/hidden states.

Measurement cost: one torch.cuda.synchronize() per call so that timings include the GPU work.
"""
import os

_LOG = os.environ.get("PDLAB_PROBE_LOG")

if _LOG:
    import importlib.abc
    import importlib.util
    import json
    import sys
    import threading
    import time

    _TARGET = "vllm.distributed.kv_transfer.kv_connector.simple_connector"
    _lock = threading.Lock()

    def _emit(rec):
        rec["ts"] = time.time()
        rec["pid"] = os.getpid()
        try:
            with _lock, open(_LOG, "a") as f:
                f.write(json.dumps(rec) + "\n")
        except Exception:
            pass

    def _nb(t):
        try:
            return int(t.numel() * t.element_size())
        except Exception:
            return 0

    def _sync():
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass

    def _patch(mod):
        cls = mod.SimpleConnector
        o_select, o_insert = cls.select, cls.insert
        o_recv, o_send = cls.recv_kv_caches_and_hidden_states, cls.send_kv_caches_and_hidden_states

        def _safe(build):
            """Bookkeeping must never raise into vLLM."""
            try:
                _emit(build())
            except Exception as e:
                _emit({"ev": "probe_error", "error": repr(e)[:300]})

        def _seq_lens(args, kwargs):
            mi = kwargs.get("model_input", args[1] if len(args) > 1 else None)
            try:
                return list(mi.attn_metadata.seq_lens or [])
            except Exception:
                return []

        def select(self, input_tokens, roi):
            t0 = time.perf_counter()
            ret = o_select(self, input_tokens, roi)
            _sync()
            ms = round((time.perf_counter() - t0) * 1000, 4)

            def build():
                hit = ret is not None and ret[0] is not None
                rec = {"ev": "select", "hit": bool(hit), "n_tokens": int(input_tokens.numel()), "ms": ms}
                if hit:
                    rec.update({"matched_tokens": int(ret[1].shape[0]), "bytes_k": _nb(ret[2]), "bytes_v": _nb(ret[3]),
                                "bytes_hidden": _nb(ret[4]) if ret[4] is not None else 0,
                                "k_shape": list(ret[2].shape), "dtype": str(getattr(ret[2], "dtype", ""))})
                return rec
            _safe(build)
            return ret

        def insert(self, input_tokens, roi, key, value, hidden):
            t0 = time.perf_counter()
            o_insert(self, input_tokens, roi, key, value, hidden)
            ms = round((time.perf_counter() - t0) * 1000, 4)
            _safe(lambda: {"ev": "insert", "n_tokens": int(input_tokens.numel()), "bytes_k": _nb(key), "bytes_v": _nb(value),
                           "bytes_hidden": _nb(hidden), "k_shape": list(key.shape), "ms": ms})

        def recv(self, *args, **kwargs):
            t0 = time.perf_counter()
            out = o_recv(self, *args, **kwargs)
            _sync()
            ms = round((time.perf_counter() - t0) * 1000, 4)

            def build():
                seq_lens = _seq_lens(args, kwargs)
                return {"ev": "recv_batch", "bypass_model_exec": bool(out[1]), "n_reqs": len(seq_lens),
                        "n_tokens": int(sum(seq_lens)), "ms": ms}
            _safe(build)
            return out

        def send(self, *args, **kwargs):
            t0 = time.perf_counter()
            out = o_send(self, *args, **kwargs)
            _sync()
            ms = round((time.perf_counter() - t0) * 1000, 4)

            def build():
                seq_lens = _seq_lens(args, kwargs)
                return {"ev": "send_batch", "n_reqs": len(seq_lens), "n_tokens": int(sum(seq_lens)), "ms": ms}
            _safe(build)
            return out

        cls.select, cls.insert = select, insert
        cls.recv_kv_caches_and_hidden_states, cls.send_kv_caches_and_hidden_states = recv, send
        _emit({"ev": "patched", "module": mod.__name__})

    class _Finder(importlib.abc.MetaPathFinder):
        def find_spec(self, fullname, path, target=None):
            if fullname != _TARGET:
                return None
            sys.meta_path.remove(self)
            spec = importlib.util.find_spec(fullname)
            if spec is None or spec.loader is None:
                return spec
            orig_exec = spec.loader.exec_module

            def exec_module(module):
                orig_exec(module)
                try:
                    _patch(module)
                except Exception as e:  # never break the server because of the probe
                    _emit({"ev": "patch_error", "error": repr(e)})

            spec.loader.exec_module = exec_module
            return spec

    sys.meta_path.insert(0, _Finder())


In [ ]:
%%writefile /kaggle/working/kaggle-pd-pp-lab/scripts/mock_vllm_server.py
"""LOCAL TESTING ONLY - a tiny fake of vLLM's OpenAI completions server.

Used to exercise the client, proxy, launcher, runner and report generator on a
machine without a GPU. It never produces numbers that go into a real report:
every run made with --mock is tagged "mock": true and analyze_results.py refuses to
mix mock and real runs.

Roles: mixed | producer | consumer. producer/consumer emulate the PD hand-off with a
file registry so that the probe-log code paths (hit / miss) can be tested.
"""
from __future__ import annotations

import argparse
import asyncio
import hashlib
import json
import os
import tempfile
import time

from aiohttp import web

REG = os.path.join(tempfile.gettempdir(), "pdlab_mock_kv")


def key(ids) -> str:
    return hashlib.sha1(json.dumps(ids).encode()).hexdigest()


def probe(rec):
    p = os.environ.get("PDLAB_PROBE_LOG")
    if p:
        rec.update({"ts": time.time(), "pid": os.getpid(), "mock": True})
        with open(p, "a") as f:
            f.write(json.dumps(rec) + "\n")


def build(role: str, prefill_ms_per_tok: float, decode_ms: float):
    os.makedirs(REG, exist_ok=True)
    lock = asyncio.Lock()  # one "GPU": serialize prefill compute

    async def completions(req: web.Request):
        body = await req.json()
        ids = body["prompt"]
        n_in, n_out = len(ids), int(body["max_tokens"])
        k = key(ids)
        skip_prefill = False
        if role == "consumer":
            path = os.path.join(REG, k)
            hit = os.path.exists(path)
            n = n_in * 28672
            probe({"ev": "select", "hit": hit, "n_tokens": n_in, "ms": 2.0 if hit else 0.3,
                   **({"matched_tokens": n_in, "bytes_k": n // 2, "bytes_v": n // 2, "bytes_hidden": n_in * 3072} if hit else {})})
            probe({"ev": "recv_batch", "bypass_model_exec": hit, "n_reqs": 1, "n_tokens": n_in, "ms": 3.0 if hit else 0.5})
            skip_prefill = hit
            if hit:
                os.remove(path)
        async with lock:
            if not skip_prefill:
                await asyncio.sleep(n_in * prefill_ms_per_tok / 1000)
        if role == "producer":
            open(os.path.join(REG, k), "w").close()
            probe({"ev": "insert", "n_tokens": n_in, "bytes_k": n_in * 14336, "bytes_v": n_in * 14336,
                   "bytes_hidden": n_in * 3072, "ms": 0.2})
        words = [f" w{i}" for i in range(n_out)]
        usage = {"prompt_tokens": n_in, "completion_tokens": n_out, "total_tokens": n_in + n_out}
        if not body.get("stream"):
            await asyncio.sleep(decode_ms / 1000 * (n_out - 1))
            return web.json_response({"id": "cmpl-mock", "object": "text_completion", "model": body["model"],
                                      "choices": [{"index": 0, "text": "".join(words), "finish_reason": "length"}],
                                      "usage": usage})
        resp = web.StreamResponse(headers={"Content-Type": "text/event-stream"})
        await resp.prepare(req)
        for i, w in enumerate(words):
            if i:
                await asyncio.sleep(decode_ms / 1000)
            ev = {"id": "cmpl-mock", "object": "text_completion", "model": body["model"],
                  "choices": [{"index": 0, "text": w, "finish_reason": "length" if i == n_out - 1 else None}],
                  "usage": None}
            await resp.write(f"data: {json.dumps(ev)}\n\n".encode())
        if (body.get("stream_options") or {}).get("include_usage"):
            await resp.write(f"data: {json.dumps({'id': 'cmpl-mock', 'choices': [], 'usage': usage})}\n\n".encode())
        await resp.write(b"data: [DONE]\n\n")
        await resp.write_eof()
        return resp

    async def health(_):
        return web.Response(text="")

    async def models(_):
        return web.json_response({"data": [{"id": "pdlab"}]})

    app = web.Application(client_max_size=64 * 1024 * 1024)
    app.router.add_post("/v1/completions", completions)
    app.router.add_get("/health", health)
    app.router.add_get("/v1/models", models)
    return app


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, required=True)
    ap.add_argument("--role", default="mixed")
    ap.add_argument("--prefill-ms-per-tok", type=float, default=0.02)
    ap.add_argument("--decode-ms", type=float, default=2.0)
    a = ap.parse_args()
    print("Model loading took 2.9 GiB (MOCK)", flush=True)
    web.run_app(build(a.role, a.prefill_ms_per_tok, a.decode_ms), host="127.0.0.1", port=a.port, access_log=None)


### （可选）恢复上一次会话的结果
如果之前的会话中断，可把下载的 `pdpp_results_*.zip` 作为 Kaggle Dataset 添加到本 Notebook（右侧 *Add Input*），
运行下面的单元格恢复已完成的配置，之后的步骤会自动跳过它们。

In [ ]:
import zipfile
zips = sorted(glob.glob("/kaggle/input/**/pdpp_results_*.zip", recursive=True))
if zips and not glob.glob("results/runs/*.json"):
    with zipfile.ZipFile(zips[-1]) as z:
        members = [m for m in z.namelist() if m.startswith("results/") and "/state/" not in m]
        z.extractall(LAB, members=members)
    print("restored from", zips[-1], len(members), "files")
else:
    print("nothing to restore" if not zips else "results/runs already present - not overwriting")

### （可选）CPU 干跑：不消耗 GPU 额度验证 Notebook 流程
在 **Accelerator = None** 的会话里也能运行：用假的 vLLM 服务器（mock）跑完 smoke → benchmark → 报告全流程，
结果写到单独目录 `/kaggle/working/mock_results`，**不会**与真实结果混在一起，报告顶部会标注 MOCK。
GPU 会话中请跳过此单元格。

In [ ]:
MOCK_ENV = "PDLAB_RESULTS=/kaggle/working/mock_results"
!{MOCK_ENV} python3 scripts/prepare_workload.py --mock-tokenizer
!for a in A C D B; do {MOCK_ENV} python3 scripts/smoke_test.py --arch $$a --mock | head -3; done
!{MOCK_ENV} python3 scripts/run_experiments.py --mock --archs A,D --workloads prefill_heavy --conc 2
!{MOCK_ENV} python3 scripts/analyze_results.py --include-mock --no-zip | tail -2

## 1. 环境检查（必须看到 `ENV_CHECK PASS`）
应看到两张 `Tesla T4`、compute_cap `7.5`、`internet True`。同时开始会话计时（预算保护的基准）。
若 FAIL：右侧 *Settings → Accelerator* 选 **GPU T4 x2**，*Internet* 打开，然后 *Run → Restart & clear outputs* 重来。

In [ ]:
!python3 scripts/env_check.py

## 2. GPU 时间预算状态（任何时候都可以重新运行）

In [ ]:
!python3 -c "import sys; sys.path.insert(0, 'scripts'); from common import Budget, load_config; print(Budget(load_config()).report())"

## 3. 安装隔离的 vLLM 环境（P0，约 5–10 分钟，只需一次）
在 `/tmp/pdlab-venv` 用 uv 安装 `vllm==0.8.5.post1`，所有依赖按 2025-05-20 的版本解析（不改动 Kaggle 系统 Python）。
必须看到 `INSTALL_STEP env PASS`，且输出中 `devices` 为两张 cc `[7, 5]`、`xformers` 有版本号。
失败时日志在 `results/logs/install.log`。

In [ ]:
!python3 scripts/install_or_check.py --step env

## 4. 下载模型（固定 revision）并生成所有架构共用的 token 负载

In [ ]:
!python3 scripts/install_or_check.py --step model
!{VPY} scripts/prepare_workload.py

## 5. A 组：单卡 smoke test（P0）
期望：`SMOKE A ... PASS`，日志行 `engine=V0 0.8.5.post1 tp=1 pp=1`，`attn=['XFormers']`，GPU0 显存约 12 GB、GPU1 接近 0。

In [ ]:
!{VPY} scripts/smoke_test.py --arch A

## 6. C 组：PP2 smoke test（P1）
期望：`tp=1 pp=2`，`weights_GiB` 有 **两个** 值，每个都明显小于 A 的单卡权重（约 55–60%：一半的层 + 各自一份 tied embedding/lm_head），证明是层切分而不是 TP2 或两个副本；两张卡都占用显存。

In [ ]:
!{VPY} scripts/smoke_test.py --arch C

## 7. D 组：PD 分离 + 真实 KV 传输 smoke test（P2，最关键）
期望：
- `PD transfer: hits=8/8 bypass_false=0`：每个经代理的请求，decode 实例都从 prefill 实例收到了 KV，且 **没有重算 prefill**（vLLM 官方 `bypass_model_exec=True`）；
- `bytes/token=28672`：与理论值 28 层×2×2 头×128×2B 一致；
- 负对照（绕过 prefill 直接发给 decode）出现 `n_select_miss: 1, n_recv_bypass_false: 1`，说明探针能区分“真的传了”和“重算了”；
- decode 日志里官方 DEBUG 行 `kv_ok` 计数 ≥ 8，`kv_fail` 只来自负对照。

In [ ]:
!{VPY} scripts/smoke_test.py --arch D

## 8. B 组：DP2 smoke test（P3）
期望：两个实例各加载一次完整权重，代理把请求分到两个副本（`per_backend` 两个都 > 0）。

In [ ]:
!{VPY} scripts/smoke_test.py --arch B

## 9. 正式小规模 benchmark
只运行 smoke 为 PASS 的架构；顺序 A → C → D → B；每个架构启动一次、跑完 2 负载 × 2 并发后立即关闭并确认显存释放。
每个配置完成即写入 `results/runs/`，中断后重新运行本单元格会自动跳过已完成配置。

In [ ]:
!{VPY} scripts/run_experiments.py --archs A,C,D,B --reps 1

### 9b.（可选，仅在时间充裕时）第二次重复
只会补跑 rep 2（rep 1 跳过），约再用 30–40 分钟。先看第 2 步的预算状态再决定。

In [ ]:
# !{VPY} scripts/run_experiments.py --archs A,C,D,B --reps 2

## 10. 汇总：报告、图表、ZIP

In [ ]:
!python3 scripts/analyze_results.py
from IPython.display import Markdown, Image, display
display(Markdown(open("results/EXPERIMENT_REPORT.md", encoding="utf-8").read()))
for f in sorted(glob.glob("results/figures/*.png")):
    display(Image(f))
print(sorted(glob.glob("/kaggle/working/pdpp_results_*.zip")))

## 11. 结束：关闭所有服务并释放 GPU（务必执行）
运行后确认两张卡显存都接近 0。**然后在 Kaggle 页面右上角点击 Stop Session（或 Run → Stop session）**，否则 GPU 额度会继续计时。
下载结果：右侧 *Output*（/kaggle/working）里找到 `pdpp_results_*.zip`，点击下载。

In [ ]:
!python3 scripts/servers.py stop
!nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv
print("REMINDER: now click 'Stop Session' in the Kaggle UI to stop GPU quota usage.")

## 附：排错工具

In [ ]:
# 最近的服务器日志（把 A/B/C/D 和实例名换成需要的）
!ls -la results/logs/
!tail -n 60 results/logs/server_D_decode.log
!tail -n 5 results/runs/_failures.jsonl 2>/dev/null
!python3 scripts/servers.py status

In [ ]:
# 随时打包当前 results（不做分析），防止会话意外结束丢数据
import shutil, time
p = shutil.make_archive(f"/kaggle/working/pdpp_partial_{time.strftime('%H%M%S')}", "zip", LAB, "results")
print(p)